In [1]:
## ULTIMATE Comprehensive Dataset Builder
## Includes EVERYTHING: Floor Pivots, Overnight touches, Initial Balance, ALL features

import pandas as pd
import numpy as np
from datetime import datetime

# Load the data
file_path = r"C:\Users\tammm\NQ and ES analysis\nq_15y_1min.csv"
df = pd.read_csv(file_path)

# Combine Date and Time into a datetime column
df['Datetime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'])
df.set_index('Datetime', inplace=True)

# Filter data
start_date = pd.to_datetime('2013-01-01')
end_date = pd.to_datetime('2025-12-31')
df = df.loc[start_date:end_date]

# Adjust the index to group by NY trading day
df['NY_Trading_Day'] = (df.index - pd.Timedelta(hours=7)).normalize()
dates = df['NY_Trading_Day'].unique()

print("="*80)
print("ULTIMATE COMPREHENSIVE DATASET BUILDER")
print("="*80)
print(f"Analyzing {len(dates)-1} trading days from {start_date.date()} to {end_date}")
print("\nFeatures being captured:")
print("  ✓ Floor Pivots (PP, R1-R3, S1-S3, Golden Zones)")
print("  ✓ Opening position relative to floor pivots")
print("  ✓ Overnight pivot touches (which levels hit before 9:30)")
print("  ✓ Previous day/week high/low touches during overnight")
print("  ✓ All hourly ranges + 4H candles (pre-market only)")
print("  ✓ All session patterns + ORBs")
print("  ✓ Initial Balance (9:30-10:30am)")
print("  ✓ NY ORB (9:30-9:45am) + 9:35 ORB (9:30-9:35am)")
print("  ✓ Level touches before 10:30am (prev day/week high/low)")
print("  ✓ Gap analysis vs prev close")
print("  NOTE: 10am-2pm and 2pm-5pm candles excluded (too late for prediction)")
print()

# ============================================================
# FLOOR PIVOT FUNCTIONS
# ============================================================

def calculate_floor_pivots(high, low, close):
    """Calculate floor pivot levels"""
    pp = (high + low + close) / 3
    
    r1 = (2 * pp) - low
    r2 = pp + (high - low)
    r3 = high + 2 * (pp - low)
    
    s1 = (2 * pp) - high
    s2 = pp - (high - low)
    s3 = low - 2 * (high - pp)
    
    # Golden Zone (0.5-0.618 of previous day range)
    pivot_range = high - low
    gz_low = low + pivot_range * 0.5
    gz_high = low + pivot_range * 0.618
    
    return {
        'PP': pp, 'R1': r1, 'R2': r2, 'R3': r3,
        'S1': s1, 'S2': s2, 'S3': s3,
        'GZ_LOW': gz_low, 'GZ_HIGH': gz_high
    }

def get_pivot_position(price, pivots, threshold_pct=0.1):
    """Determine position relative to floor pivots"""
    threshold = pivots['PP'] * (threshold_pct / 100)
    
    if abs(price - pivots['PP']) <= threshold:
        return 'at_pp'
    elif pivots['GZ_LOW'] <= price <= pivots['GZ_HIGH']:
        return 'in_gz'
    elif price >= pivots['R3']:
        return 'above_r3'
    elif price >= pivots['R2']:
        return 'r2_to_r3'
    elif price >= pivots['R1']:
        return 'r1_to_r2'
    elif price > pivots['GZ_HIGH']:
        return 'gz_to_r1'
    elif price > pivots['PP']:
        return 'pp_to_gz'
    elif price <= pivots['S3']:
        return 'below_s3'
    elif price <= pivots['S2']:
        return 's3_to_s2'
    elif price <= pivots['S1']:
        return 's2_to_s1'
    elif price < pivots['GZ_LOW']:
        return 's1_to_gz'
    else:
        return 'gz_to_pp'

def check_level_touch(data, level_price, threshold_pct=0.05):
    """Check if a price level was touched in given data"""
    if data.empty:
        return False
    threshold = level_price * (threshold_pct / 100)
    for _, row in data.iterrows():
        if row['Low'] - threshold <= level_price <= row['High'] + threshold:
            return True
    return False

# ============================================================
# BUILD COMPREHENSIVE DATASET
# ============================================================

unified_data = []
skipped_count = 0

for idx, date in enumerate(dates[1:], 1):
    if idx % 500 == 0:
        print(f"  Processing day {idx}...")
    
    actual_date = date + pd.Timedelta(hours=7)
    
    # Define trading day boundaries
    ny_trading_day_start = actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
    ny_trading_day_end = actual_date.replace(hour=17, minute=0, second=0)
    
    full_day_data = df.loc[ny_trading_day_start:ny_trading_day_end]
    if full_day_data.empty:
        skipped_count += 1
        continue
    
    # ========================================
    # PREVIOUS DAY DATA
    # ========================================
    prev_date = date - pd.Timedelta(days=1)
    prev_actual_date = prev_date + pd.Timedelta(hours=7)
    prev_day_start = prev_actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
    prev_day_end = prev_actual_date.replace(hour=17, minute=0, second=0)
    prev_day_data = df.loc[prev_day_start:prev_day_end]
    
    if not prev_day_data.empty:
        prev_day_high = prev_day_data['High'].max()
        prev_day_low = prev_day_data['Low'].min()
        prev_day_open = prev_day_data['Open'].iloc[0]
        prev_day_close = prev_day_data['Close'].iloc[-1]
        prev_day_direction = 'bullish' if prev_day_close > prev_day_open else 'bearish'
    else:
        prev_day_high = prev_day_low = prev_day_open = prev_day_close = prev_day_direction = None
    
    # ========================================
    # PREVIOUS WEEK DATA
    # ========================================
    current_day_index = list(dates).index(date)
    if current_day_index >= 5:
        prev_week_dates = dates[current_day_index-5:current_day_index]
        prev_week_data_list = []
        
        for pwd in prev_week_dates:
            pwd_actual = pwd + pd.Timedelta(hours=7)
            pwd_start = pwd_actual.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
            pwd_end = pwd_actual.replace(hour=17, minute=0, second=0)
            pwd_data = df.loc[pwd_start:pwd_end]
            if not pwd_data.empty:
                prev_week_data_list.append(pwd_data)
        
        if prev_week_data_list:
            prev_week_combined = pd.concat(prev_week_data_list)
            prev_week_high = prev_week_combined['High'].max()
            prev_week_low = prev_week_combined['Low'].min()
        else:
            prev_week_high = prev_week_low = None
    else:
        prev_week_high = prev_week_low = None
    
    # ========================================
    # FLOOR PIVOTS (from previous day)
    # ========================================
    if prev_day_high and prev_day_low and prev_day_close:
        floor_pivots = calculate_floor_pivots(prev_day_high, prev_day_low, prev_day_close)
    else:
        floor_pivots = None
    
    # ========================================
    # OVERNIGHT SESSION (6pm-9:30am)
    # ========================================
    overnight_start = ny_trading_day_start
    overnight_end = actual_date.replace(hour=9, minute=30, second=0)
    overnight_data = df.loc[overnight_start:overnight_end]
    
    if not overnight_data.empty:
        overnight_high = overnight_data['High'].max()
        overnight_low = overnight_data['Low'].min()
        overnight_open = overnight_data['Open'].iloc[0]
        overnight_close = overnight_data['Close'].iloc[-1]
        overnight_direction = 'bullish' if overnight_close > overnight_open else 'bearish'
        
        # Check if prev day/week levels touched overnight
        overnight_touched_prev_day_high = check_level_touch(overnight_data, prev_day_high) if prev_day_high else False
        overnight_touched_prev_day_low = check_level_touch(overnight_data, prev_day_low) if prev_day_low else False
        overnight_touched_prev_week_high = check_level_touch(overnight_data, prev_week_high) if prev_week_high else False
        overnight_touched_prev_week_low = check_level_touch(overnight_data, prev_week_low) if prev_week_low else False
        
        # Check which floor pivots touched overnight
        if floor_pivots:
            overnight_touched_r3 = check_level_touch(overnight_data, floor_pivots['R3'])
            overnight_touched_r2 = check_level_touch(overnight_data, floor_pivots['R2'])
            overnight_touched_r1 = check_level_touch(overnight_data, floor_pivots['R1'])
            overnight_touched_pp = check_level_touch(overnight_data, floor_pivots['PP'])
            overnight_touched_s1 = check_level_touch(overnight_data, floor_pivots['S1'])
            overnight_touched_s2 = check_level_touch(overnight_data, floor_pivots['S2'])
            overnight_touched_s3 = check_level_touch(overnight_data, floor_pivots['S3'])
            overnight_touched_gz = (overnight_data['Low'].min() <= floor_pivots['GZ_HIGH'] and 
                                   overnight_data['High'].max() >= floor_pivots['GZ_LOW'])
            
            # Highest/lowest pivot touched overnight
            overnight_highest_pivot = None
            overnight_lowest_pivot = None
            for level in ['R3', 'R2', 'R1']:
                if check_level_touch(overnight_data, floor_pivots[level]):
                    overnight_highest_pivot = level
                    break
            for level in ['S3', 'S2', 'S1']:
                if check_level_touch(overnight_data, floor_pivots[level]):
                    overnight_lowest_pivot = level
                    break
        else:
            overnight_touched_r3 = overnight_touched_r2 = overnight_touched_r1 = False
            overnight_touched_pp = overnight_touched_s1 = overnight_touched_s2 = overnight_touched_s3 = False
            overnight_touched_gz = False
            overnight_highest_pivot = overnight_lowest_pivot = None
    else:
        overnight_high = overnight_low = overnight_open = overnight_direction = None
        overnight_touched_prev_day_high = overnight_touched_prev_day_low = False
        overnight_touched_prev_week_high = overnight_touched_prev_week_low = False
        overnight_touched_r3 = overnight_touched_r2 = overnight_touched_r1 = False
        overnight_touched_pp = overnight_touched_s1 = overnight_touched_s2 = overnight_touched_s3 = False
        overnight_touched_gz = False
        overnight_highest_pivot = overnight_lowest_pivot = None
    
    # ========================================
    # ASIA SESSION
    # ========================================
    asian_start = actual_date.replace(hour=20, minute=0, second=0) - pd.Timedelta(days=1)
    asian_end = actual_date.replace(hour=2, minute=0, second=0)
    asian_data = df.loc[asian_start:asian_end]
    
    if not asian_data.empty:
        asia_high = asian_data['High'].max()
        asia_low = asian_data['Low'].min()
        asia_open = asian_data['Open'].iloc[0]
        asia_close = asian_data['Close'].iloc[-1]
        asia_direction = 'bullish' if asia_close > asia_open else 'bearish'
    else:
        asia_high = asia_low = asia_open = asia_close = asia_direction = None
    
    # ========================================
    # LONDON SESSION
    # ========================================
    london_start = actual_date.replace(hour=2, minute=0, second=0)
    london_end = actual_date.replace(hour=8, minute=0, second=0)
    london_data = df.loc[london_start:london_end]
    
    if not london_data.empty:
        london_high = london_data['High'].max()
        london_low = london_data['Low'].min()
        london_open = london_data['Open'].iloc[0]
        london_close = london_data['Close'].iloc[-1]
        london_direction = 'bullish' if london_close > london_open else 'bearish'
    else:
        london_high = london_low = london_open = london_close = london_direction = None
    
    # ========================================
    # SESSION PATTERN
    # ========================================
    if asia_high and london_high:
        if london_high > asia_high and london_low < asia_low:
            session_pattern = 'london_engulfs_asia'
        elif london_high <= asia_high and london_low >= asia_low:
            session_pattern = 'asia_engulfs_london'
        elif london_high > asia_high and london_low >= asia_low:
            session_pattern = 'london_partial_up'
        elif london_low < asia_low and london_high <= asia_high:
            session_pattern = 'london_partial_down'
        else:
            session_pattern = 'other'
    else:
        session_pattern = None
    
    # ========================================
    # HOURLY RANGES
    # ========================================
    hourly_ranges = {}
    
    # Globex 6pm
    globex_start = actual_date.replace(hour=18, minute=0, second=0) - pd.Timedelta(days=1)
    globex_end = globex_start + pd.Timedelta(hours=1)
    globex_data = df.loc[globex_start:globex_end]
    if not globex_data.empty:
        hourly_ranges['globex_6pm'] = {
            'direction': 'bullish' if globex_data['Close'].iloc[-1] > globex_data['Open'].iloc[0] else 'bearish'
        }
    else:
        hourly_ranges['globex_6pm'] = None
    
    # Asia 8pm
    asia_hour_start = actual_date.replace(hour=20, minute=0, second=0) - pd.Timedelta(days=1)
    asia_hour_end = asia_hour_start + pd.Timedelta(hours=1)
    asia_hour_data = df.loc[asia_hour_start:asia_hour_end]
    if not asia_hour_data.empty:
        hourly_ranges['asia_8pm'] = {
            'direction': 'bullish' if asia_hour_data['Close'].iloc[-1] > asia_hour_data['Open'].iloc[0] else 'bearish'
        }
    else:
        hourly_ranges['asia_8pm'] = None
    
    # Midnight
    midnight_start = actual_date.replace(hour=0, minute=0, second=0)
    midnight_end = midnight_start + pd.Timedelta(hours=1)
    midnight_data = df.loc[midnight_start:midnight_end]
    if not midnight_data.empty:
        hourly_ranges['midnight'] = {
            'direction': 'bullish' if midnight_data['Close'].iloc[-1] > midnight_data['Open'].iloc[0] else 'bearish'
        }
    else:
        hourly_ranges['midnight'] = None
    
    # London 2am
    london_hour_start = actual_date.replace(hour=2, minute=0, second=0)
    london_hour_end = london_hour_start + pd.Timedelta(hours=1)
    london_hour_data = df.loc[london_hour_start:london_hour_end]
    if not london_hour_data.empty:
        hourly_ranges['london_2am'] = {
            'direction': 'bullish' if london_hour_data['Close'].iloc[-1] > london_hour_data['Open'].iloc[0] else 'bearish'
        }
    else:
        hourly_ranges['london_2am'] = None
    
    # 7am
    hour_7am_start = actual_date.replace(hour=7, minute=0, second=0)
    hour_7am_end = hour_7am_start + pd.Timedelta(hours=1)
    hour_7am_data = df.loc[hour_7am_start:hour_7am_end]
    if not hour_7am_data.empty:
        hourly_ranges['hour_7am'] = {
            'direction': 'bullish' if hour_7am_data['Close'].iloc[-1] > hour_7am_data['Open'].iloc[0] else 'bearish'
        }
    else:
        hourly_ranges['hour_7am'] = None
    
    # 8am
    hour_8am_start = actual_date.replace(hour=8, minute=0, second=0)
    hour_8am_end = hour_8am_start + pd.Timedelta(hours=1)
    hour_8am_data = df.loc[hour_8am_start:hour_8am_end]
    if not hour_8am_data.empty:
        hourly_ranges['ny_futures_8am'] = {
            'direction': 'bullish' if hour_8am_data['Close'].iloc[-1] > hour_8am_data['Open'].iloc[0] else 'bearish'
        }
    else:
        hourly_ranges['ny_futures_8am'] = None
    
    # ========================================
    # 4H CANDLES (Only those complete before useful entry times)
    # ========================================
    four_hour_candles = {}
    
    candle_configs = [
        ('6pm', 18, 22, -1),   # 6pm-10pm previous day - complete by 10pm
        ('10pm', 22, 26, -1),  # 10pm-2am - complete by 2am
        ('2am', 2, 6, 0),      # 2am-6am - complete by 6am
        ('6am', 6, 10, 0),     # 6am-10am - complete by 10am (for 10:30+ entries only)
        # REMOVED: ('10am', 10, 14, 0) - completes at 2pm, too late for prediction
        # REMOVED: ('2pm', 14, 17, 0) - completes at 5pm, way too late
    ]
    
    for candle_name, start_hour, end_hour, day_offset in candle_configs:
        base_date = actual_date + pd.Timedelta(days=day_offset)
        candle_start = base_date.replace(hour=start_hour, minute=0, second=0)
        
        if end_hour >= 24:
            candle_end = (base_date + pd.Timedelta(days=1)).replace(hour=end_hour-24, minute=0, second=0)
        else:
            candle_end = base_date.replace(hour=end_hour, minute=0, second=0)
        
        candle_data = df.loc[candle_start:candle_end]
        
        if not candle_data.empty:
            c_open = candle_data['Open'].iloc[0]
            c_close = candle_data['Close'].iloc[-1]
            four_hour_candles[f'candle_4h_{candle_name}'] = {
                'direction': 'bullish' if c_close > c_open else 'bearish'
            }
        else:
            four_hour_candles[f'candle_4h_{candle_name}'] = None
    
    # ========================================
    # 6-9AM RANGE
    # ========================================
    range_6_9_start = actual_date.replace(hour=6, minute=0, second=0)
    range_6_9_end = actual_date.replace(hour=9, minute=0, second=0)
    range_6_9_data = df.loc[range_6_9_start:range_6_9_end]
    
    if not range_6_9_data.empty:
        r69_open = range_6_9_data['Open'].iloc[0]
        r69_close = range_6_9_data['Close'].iloc[-1]
        range_6_9_direction = 'bullish' if r69_close > r69_open else 'bearish'
    else:
        range_6_9_direction = None
    
    # ========================================
    # PRE-MARKET ORBS (before 9:30)
    # ========================================
    orb_periods = {}
    
    # Globex ORB (6pm+15)
    if hourly_ranges['globex_6pm']:
        orb_globex_start = globex_start
        orb_globex_end = orb_globex_start + pd.Timedelta(minutes=15)
        orb_globex_data = df.loc[orb_globex_start:orb_globex_end]
        if len(orb_globex_data) >= 10:
            orb_periods['globex_orb'] = {
                'direction': 'bullish' if orb_globex_data['Close'].iloc[-1] > orb_globex_data['Open'].iloc[0] else 'bearish'
            }
        else:
            orb_periods['globex_orb'] = None
    else:
        orb_periods['globex_orb'] = None
    
    # Asia ORB (8pm+15)
    if hourly_ranges['asia_8pm']:
        orb_asia_start = asia_hour_start
        orb_asia_end = orb_asia_start + pd.Timedelta(minutes=15)
        orb_asia_data = df.loc[orb_asia_start:orb_asia_end]
        if len(orb_asia_data) >= 10:
            orb_periods['asia_orb'] = {
                'direction': 'bullish' if orb_asia_data['Close'].iloc[-1] > orb_asia_data['Open'].iloc[0] else 'bearish'
            }
        else:
            orb_periods['asia_orb'] = None
    else:
        orb_periods['asia_orb'] = None
    
    # London ORB (2am+15)
    if hourly_ranges['london_2am']:
        orb_london_start = london_hour_start
        orb_london_end = orb_london_start + pd.Timedelta(minutes=15)
        orb_london_data = df.loc[orb_london_start:orb_london_end]
        if len(orb_london_data) >= 10:
            orb_periods['london_orb'] = {
                'direction': 'bullish' if orb_london_data['Close'].iloc[-1] > orb_london_data['Open'].iloc[0] else 'bearish'
            }
        else:
            orb_periods['london_orb'] = None
    else:
        orb_periods['london_orb'] = None
    
    # PreMarket ORB (8am+15)
    if hourly_ranges['ny_futures_8am']:
        orb_premarket_start = hour_8am_start
        orb_premarket_end = orb_premarket_start + pd.Timedelta(minutes=15)
        orb_premarket_data = df.loc[orb_premarket_start:orb_premarket_end]
        if len(orb_premarket_data) >= 10:
            orb_periods['premarket_orb'] = {
                'direction': 'bullish' if orb_premarket_data['Close'].iloc[-1] > orb_premarket_data['Open'].iloc[0] else 'bearish'
            }
        else:
            orb_periods['premarket_orb'] = None
    else:
        orb_periods['premarket_orb'] = None
    
    # ========================================
    # NY CASH SESSION (9:30-4pm)
    # ========================================
    ny_cash_start = actual_date.replace(hour=9, minute=30, second=0)
    ny_cash_end = actual_date.replace(hour=16, minute=0, second=0)
    ny_cash_data = df.loc[ny_cash_start:ny_cash_end]
    
    if not ny_cash_data.empty:
        ny_open = ny_cash_data['Open'].iloc[0]
        ny_close = ny_cash_data['Close'].iloc[-1]
        ny_high = ny_cash_data['High'].max()
        ny_low = ny_cash_data['Low'].min()
        ny_direction = 'bullish' if ny_close > ny_open else 'bearish'
        ny_range_pct = ((ny_high - ny_low) / ny_low) * 100
    else:
        ny_open = ny_close = ny_high = ny_low = ny_direction = ny_range_pct = None
    
    # ========================================
    # NY ORB (9:30-9:45) - for POST-ORB predictions
    # ========================================
    ny_orb_start = actual_date.replace(hour=9, minute=30, second=0)
    ny_orb_end = actual_date.replace(hour=9, minute=45, second=0)
    ny_orb_data = df.loc[ny_orb_start:ny_orb_end]
    
    if len(ny_orb_data) >= 10:
        orb_high = ny_orb_data['High'].max()
        orb_low = ny_orb_data['Low'].min()
        orb_open = ny_orb_data['Open'].iloc[0]
        orb_close = ny_orb_data['Close'].iloc[-1]
        ny_orb_direction = 'bullish' if orb_close > orb_open else 'bearish'
        
        # Check which side breaks first
        rest_of_day_start = ny_orb_end
        rest_of_day_end = actual_date.replace(hour=16, minute=0, second=0)
        rest_of_day = df.loc[rest_of_day_start:rest_of_day_end]
        
        orb_first_break = None
        if not rest_of_day.empty:
            for idx, row in rest_of_day.iterrows():
                if orb_first_break is None:
                    if row['High'] > orb_high:
                        orb_first_break = 'high'
                    elif row['Low'] < orb_low:
                        orb_first_break = 'low'
                if orb_first_break:
                    break
    else:
        ny_orb_direction = orb_first_break = None
    
    # ========================================
    # 9:30-9:35 ORB (5-minute) - for 9:45+ predictions
    # ========================================
    orb_935_start = actual_date.replace(hour=9, minute=30, second=0)
    orb_935_end = actual_date.replace(hour=9, minute=35, second=0)
    orb_935_data = df.loc[orb_935_start:orb_935_end]
    
    if len(orb_935_data) >= 3:
        orb935_high = orb_935_data['High'].max()
        orb935_low = orb_935_data['Low'].min()
        orb935_open = orb_935_data['Open'].iloc[0]
        orb935_close = orb_935_data['Close'].iloc[-1]
        orb_935_direction = 'bullish' if orb935_close > orb935_open else 'bearish'
        
        # Check which side breaks first after 9:35
        rest_after_935_start = orb_935_end
        rest_after_935_end = actual_date.replace(hour=16, minute=0, second=0)
        rest_after_935 = df.loc[rest_after_935_start:rest_after_935_end]
        
        orb_935_first_break = None
        if not rest_after_935.empty:
            for idx, row in rest_after_935.iterrows():
                if orb_935_first_break is None:
                    if row['High'] > orb935_high:
                        orb_935_first_break = 'high'
                    elif row['Low'] < orb935_low:
                        orb_935_first_break = 'low'
                if orb_935_first_break:
                    break
    else:
        orb_935_direction = orb_935_first_break = None
    
    # ========================================
    # INITIAL BALANCE (9:30-10:30) - for POST-IB predictions
    # ========================================
    ib_start = actual_date.replace(hour=9, minute=30, second=0)
    ib_end = actual_date.replace(hour=10, minute=30, second=0)
    ib_data = df.loc[ib_start:ib_end]
    
    if not ib_data.empty:
        ib_high = ib_data['High'].max()
        ib_low = ib_data['Low'].min()
        ib_open = ib_data['Open'].iloc[0]
        ib_close = ib_data['Close'].iloc[-1]
        ib_direction = 'bullish' if ib_close > ib_open else 'bearish'
        ib_range_pct = ((ib_high - ib_low) / ib_low) * 100
    else:
        ib_high = ib_low = ib_open = ib_direction = ib_range_pct = None
    
    # ========================================
    # TOUCHES BEFORE 10:30AM (for 10:30+ predictions)
    # ========================================
    before_1030_start = ny_trading_day_start
    before_1030_end = actual_date.replace(hour=10, minute=30, second=0)
    before_1030_data = df.loc[before_1030_start:before_1030_end]
    
    touched_prev_day_high_before_1030 = False
    touched_prev_day_low_before_1030 = False
    touched_prev_week_high_before_1030 = False
    touched_prev_week_low_before_1030 = False
    
    if not before_1030_data.empty and prev_day_high and prev_day_low:
        # Check if prev day high/low was touched
        touched_prev_day_high_before_1030 = check_level_touch(before_1030_data, prev_day_high, threshold_pct=0.05)
        touched_prev_day_low_before_1030 = check_level_touch(before_1030_data, prev_day_low, threshold_pct=0.05)
    
    if not before_1030_data.empty and prev_week_high and prev_week_low:
        # Check if prev week high/low was touched
        touched_prev_week_high_before_1030 = check_level_touch(before_1030_data, prev_week_high, threshold_pct=0.05)
        touched_prev_week_low_before_1030 = check_level_touch(before_1030_data, prev_week_low, threshold_pct=0.05)
    
    # ========================================
    # GAP ANALYSIS
    # ========================================
    if ny_open and prev_day_close:
        gap_pct = ((ny_open - prev_day_close) / prev_day_close) * 100
        
        if gap_pct > 0.1:
            gap_type = 'gap_up'
        elif gap_pct < -0.1:
            gap_type = 'gap_down'
        else:
            gap_type = 'flat'
        
        abs_gap = abs(gap_pct)
        if abs_gap < 0.25:
            gap_size = 'small'
        elif abs_gap < 0.75:
            gap_size = 'medium'
        else:
            gap_size = 'large'
    else:
        gap_pct = gap_type = gap_size = None
    
    # ========================================
    # FLOOR PIVOT POSITION AT NY OPEN
    # ========================================
    if floor_pivots and ny_open:
        ny_open_pivot_position = get_pivot_position(ny_open, floor_pivots)
    else:
        ny_open_pivot_position = None
    
    # ========================================
    # POSITIONING
    # ========================================
    if ny_open and prev_day_high:
        if ny_open > prev_day_high:
            ny_open_vs_prev_day = 'above_high'
        elif ny_open < prev_day_low:
            ny_open_vs_prev_day = 'below_low'
        elif ny_open > prev_day_open:
            ny_open_vs_prev_day = 'above_open'
        elif ny_open < prev_day_open:
            ny_open_vs_prev_day = 'below_open'
        else:
            ny_open_vs_prev_day = 'at_open'
    else:
        ny_open_vs_prev_day = None
    
    if ny_open and prev_week_high:
        if ny_open > prev_week_high:
            ny_open_vs_prev_week = 'above_high'
        elif ny_open < prev_week_low:
            ny_open_vs_prev_week = 'below_low'
        else:
            ny_open_vs_prev_week = 'within'
    else:
        ny_open_vs_prev_week = None
    
    if ny_open and asia_high and london_high:
        overnight_high_combined = max(asia_high, london_high)
        overnight_low_combined = min(asia_low, london_low)
        
        if ny_open > overnight_high_combined:
            ny_open_vs_overnight = 'above'
        elif ny_open < overnight_low_combined:
            ny_open_vs_overnight = 'below'
        else:
            ny_open_vs_overnight = 'within'
    else:
        ny_open_vs_overnight = None
    
    # ========================================
    # DAY OF WEEK
    # ========================================
    day_of_week = actual_date.strftime('%A')
    
    # ========================================
    # VALIDATION
    # ========================================
    required_fields = [
        session_pattern, asia_direction, london_direction, 
        ny_direction, ny_open_vs_overnight
    ]
    
    if all(x is not None for x in required_fields):
        record = {
            'date': actual_date.date(),
            'day_of_week': day_of_week,
            
            # Previous context
            'prev_day_direction': prev_day_direction,
            'prev_day_close': prev_day_close,
            
            # Gap analysis
            'gap_type': gap_type,
            'gap_size': gap_size,
            'gap_pct': gap_pct,
            
            # Positioning
            'ny_open_vs_prev_day': ny_open_vs_prev_day,
            'ny_open_vs_prev_week': ny_open_vs_prev_week,
            'ny_open_vs_overnight': ny_open_vs_overnight,
            
            # Floor pivots
            'ny_open_pivot_position': ny_open_pivot_position,
            'overnight_highest_pivot': overnight_highest_pivot,
            'overnight_lowest_pivot': overnight_lowest_pivot,
            'overnight_touched_pp': overnight_touched_pp,
            'overnight_touched_gz': overnight_touched_gz,
            'overnight_touched_r1': overnight_touched_r1,
            'overnight_touched_r2': overnight_touched_r2,
            'overnight_touched_r3': overnight_touched_r3,
            'overnight_touched_s1': overnight_touched_s1,
            'overnight_touched_s2': overnight_touched_s2,
            'overnight_touched_s3': overnight_touched_s3,
            
            # Overnight level touches
            'overnight_touched_prev_day_high': overnight_touched_prev_day_high,
            'overnight_touched_prev_day_low': overnight_touched_prev_day_low,
            'overnight_touched_prev_week_high': overnight_touched_prev_week_high,
            'overnight_touched_prev_week_low': overnight_touched_prev_week_low,
            'overnight_direction': overnight_direction,
            
            # Session patterns
            'session_pattern': session_pattern,
            'asia_direction': asia_direction,
            'london_direction': london_direction,
            
            # NY ORB (for post-9:45 predictions)
            'ny_orb_direction': ny_orb_direction,
            'ny_orb_first_break': orb_first_break,
            
            # 9:30-9:35 ORB (5-minute, for post-9:45 predictions)
            'orb_935_direction': orb_935_direction,
            'orb_935_first_break': orb_935_first_break,
            
            # Initial Balance (for post-10:30 predictions)
            'ib_direction': ib_direction,
            'ib_range_pct': ib_range_pct,
            
            # Touches before 10:30am (for post-10:30 predictions)
            'touched_prev_day_high_before_1030': touched_prev_day_high_before_1030,
            'touched_prev_day_low_before_1030': touched_prev_day_low_before_1030,
            'touched_prev_week_high_before_1030': touched_prev_week_high_before_1030,
            'touched_prev_week_low_before_1030': touched_prev_week_low_before_1030,
            
            # Outcome
            'ny_direction': ny_direction,
            'ny_range_pct': ny_range_pct,
        }
        
        # Add hourly ranges
        for hour_name, hour_data in hourly_ranges.items():
            if hour_data:
                record[f'{hour_name}_direction'] = hour_data['direction']
            else:
                record[f'{hour_name}_direction'] = None
        
        # Add 4H candles
        for candle_name, candle_data in four_hour_candles.items():
            if candle_data:
                record[f'{candle_name}_direction'] = candle_data['direction']
            else:
                record[f'{candle_name}_direction'] = None
        
        # Add 6-9am range
        record['range_6_9_direction'] = range_6_9_direction
        
        # Add ORB periods
        for orb_name, orb_data in orb_periods.items():
            if orb_data:
                record[f'{orb_name}_direction'] = orb_data['direction']
            else:
                record[f'{orb_name}_direction'] = None
        
        unified_data.append(record)
    else:
        skipped_count += 1

# Convert to DataFrame
unified_df = pd.DataFrame(unified_data)

print(f"\n{'='*80}")
print("DATASET CREATION COMPLETE")
print("="*80)
print(f"✓ Created dataset with {len(unified_df)} valid trading days")
print(f"  Skipped {skipped_count} days due to missing data")
print(f"\n  Total features captured: {len(unified_df.columns)}")

# Save
output_dir = r"C:\Users\tammm\NQ and ES analysis"
unified_df.to_csv(f'{output_dir}\\ultimate_comprehensive_dataset.csv', index=False)
print(f"\n✓ Saved: ultimate_comprehensive_dataset.csv")

print("\n" + "="*80)
print("FEATURE GROUPS")
print("="*80)

feature_groups = {
    'Floor Pivots': [col for col in unified_df.columns if 'pivot' in col or 'gz' in col.lower()],
    'Overnight Touches': [col for col in unified_df.columns if 'overnight_touched' in col],
    'Gap Analysis': [col for col in unified_df.columns if 'gap' in col],
    'Sessions': [col for col in unified_df.columns if any(x in col for x in ['asia', 'london', 'session'])],
    'Hourly Ranges': [col for col in unified_df.columns if any(x in col for x in ['globex', 'midnight', '7am', '8am']) and 'direction' in col],
    '4H Candles': [col for col in unified_df.columns if 'candle_4h' in col],
    'ORBs': [col for col in unified_df.columns if 'orb' in col],
    'NY Session': [col for col in unified_df.columns if col.startswith('ny_') or col.startswith('ib_')],
    'Positioning': [col for col in unified_df.columns if 'vs_' in col],
}

for group_name, features in feature_groups.items():
    if features:
        print(f"\n{group_name}: {len(features)} features")
        for feat in features:
            unique_vals = unified_df[feat].nunique()
            print(f"  • {feat:<50} ({unique_vals} values)")

ULTIMATE COMPREHENSIVE DATASET BUILDER
Analyzing 3785 trading days from 2013-01-01 to 2025-12-31 00:00:00

Features being captured:
  ✓ Floor Pivots (PP, R1-R3, S1-S3, Golden Zones)
  ✓ Opening position relative to floor pivots
  ✓ Overnight pivot touches (which levels hit before 9:30)
  ✓ Previous day/week high/low touches during overnight
  ✓ All hourly ranges + 4H candles (pre-market only)
  ✓ All session patterns + ORBs
  ✓ Initial Balance (9:30-10:30am)
  ✓ NY ORB (9:30-9:45am) + 9:35 ORB (9:30-9:35am)
  ✓ Level touches before 10:30am (prev day/week high/low)
  ✓ Gap analysis vs prev close
  NOTE: 10am-2pm and 2pm-5pm candles excluded (too late for prediction)

  Processing day 500...
  Processing day 1000...
  Processing day 1500...
  Processing day 2000...
  Processing day 2500...
  Processing day 3000...
  Processing day 3500...

DATASET CREATION COMPLETE
✓ Created dataset with 3126 valid trading days
  Skipped 659 days due to missing data

  Total features captured: 56

✓ Save

In [16]:
## baseline statistics for bullish/bearish - we want to exclude background baseline noise from the results

import pandas as pd
import numpy as np

input_dir = r"C:\Users\tammm\NQ and ES analysis"

# Load dataset
df = pd.read_csv(f'{input_dir}\\ultimate_comprehensive_dataset_10am\.csv')

print("="*100)
print("BASELINE STATISTICS - UNDERSTANDING MARKET BIAS")
print("="*100)
print(f"\nAnalyzing {len(df)} trading days from 2013-2025\n")

# ============================================================
# OVERALL BASELINE
# ============================================================
print("="*100)
print("OVERALL BASELINE")
print("="*100)

total_days = len(df[df['ny_direction'].notna()])
bullish_days = len(df[df['ny_direction'] == 'bullish'])
bearish_days = len(df[df['ny_direction'] == 'bearish'])

print(f"\nNY Session Direction:")
print(f"  Total days: {total_days}")
print(f"  Bullish: {bullish_days} ({bullish_days/total_days*100:.1f}%)")
print(f"  Bearish: {bearish_days} ({bearish_days/total_days*100:.1f}%)")
print(f"\n  ** BASELINE EXPECTATION: {bullish_days/total_days*100:.1f}% bullish **")

# ============================================================
# BASELINE BY KEY FEATURES
# ============================================================
print("\n" + "="*100)
print("BASELINE BY KEY FEATURES (What % bullish given each condition?)")
print("="*100)

features_to_check = [
    ('ny_orb_direction', 'NY ORB Direction'),
    ('ny_orb_first_break', 'NY ORB First Break'),
    ('orb_935_direction', '9:35 ORB Direction'),
    ('orb_935_first_break', '9:35 ORB First Break'),
    ('ib_direction', 'Initial Balance Direction'),
    ('overnight_direction', 'Overnight Direction'),
    ('gap_type', 'Gap Type'),
    ('gap_size', 'Gap Size'),
    ('day_of_week', 'Day of Week'),
    ('session_pattern', 'Session Pattern'),
    ('overnight_touched_r1', 'Touched R1 Overnight'),
    ('overnight_highest_pivot', 'Overnight Highest Pivot'),
]

for feature, description in features_to_check:
    if feature not in df.columns:
        continue
    
    print(f"\n{description} ({feature}):")
    print("-" * 100)
    
    feature_df = df[df[feature].notna() & df['ny_direction'].notna()]
    
    if len(feature_df) == 0:
        print("  No data available")
        continue
    
    grouped = feature_df.groupby(feature)['ny_direction'].apply(
        lambda x: (x == 'bullish').sum() / len(x) * 100
    ).sort_values(ascending=False)
    
    for value, pct in grouped.items():
        count = len(feature_df[feature_df[feature] == value])
        bullish_count = len(feature_df[(feature_df[feature] == value) & (feature_df['ny_direction'] == 'bullish')])
        value_str = str(value)
        print(f"  {value_str:30s}: {pct:5.1f}% bullish ({bullish_count:4d}/{count:4d} days)")

# ============================================================
# BASELINE FOR COMBINED CONDITIONS
# ============================================================
print("\n" + "="*100)
print("BASELINE FOR KEY COMBINATIONS")
print("="*100)

print("\n1. NY ORB Direction + First Break:")
print("-" * 100)
orb_df = df[df['ny_orb_direction'].notna() & df['ny_orb_first_break'].notna() & df['ny_direction'].notna()]
for orb_dir in ['bullish', 'bearish']:
    for first_break in ['high', 'low']:
        subset = orb_df[(orb_df['ny_orb_direction'] == orb_dir) & (orb_df['ny_orb_first_break'] == first_break)]
        if len(subset) > 0:
            bullish_pct = (subset['ny_direction'] == 'bullish').sum() / len(subset) * 100
            bullish_count = (subset['ny_direction'] == 'bullish').sum()
            print(f"  ORB {orb_dir:7s} + breaks {first_break:4s}: {bullish_pct:5.1f}% bullish ({bullish_count:4d}/{len(subset):4d} days)")

print("\n2. IB Direction + ORB First Break:")
print("-" * 100)
ib_df = df[df['ib_direction'].notna() & df['ny_orb_first_break'].notna() & df['ny_direction'].notna()]
for ib_dir in ['bullish', 'bearish']:
    for first_break in ['high', 'low']:
        subset = ib_df[(ib_df['ib_direction'] == ib_dir) & (ib_df['ny_orb_first_break'] == first_break)]
        if len(subset) > 0:
            bullish_pct = (subset['ny_direction'] == 'bullish').sum() / len(subset) * 100
            bullish_count = (subset['ny_direction'] == 'bullish').sum()
            print(f"  IB {ib_dir:7s} + ORB breaks {first_break:4s}: {bullish_pct:5.1f}% bullish ({bullish_count:4d}/{len(subset):4d} days)")

print("\n3. Overnight Touched R1 + IB Direction:")
print("-" * 100)
r1_df = df[df['overnight_touched_r1'].notna() & df['ib_direction'].notna() & df['ny_direction'].notna()]
for touched in [True, False]:
    for ib_dir in ['bullish', 'bearish']:
        subset = r1_df[(r1_df['overnight_touched_r1'] == touched) & (r1_df['ib_direction'] == ib_dir)]
        if len(subset) > 0:
            bullish_pct = (subset['ny_direction'] == 'bullish').sum() / len(subset) * 100
            bullish_count = (subset['ny_direction'] == 'bullish').sum()
            touched_str = "YES" if touched else "NO"
            print(f"  Touched R1: {touched_str:3s} + IB {ib_dir:7s}: {bullish_pct:5.1f}% bullish ({bullish_count:4d}/{len(subset):4d} days)")

# ============================================================
# 10AM-2PM CANDLE BASELINE
# ============================================================
if 'candle_10am_2pm_direction' in df.columns:
    print("\n" + "="*100)
    print("10AM-2PM CANDLE BASELINE")
    print("="*100)
    
    candle_df = df[df['candle_10am_2pm_direction'].notna()]
    total_candles = len(candle_df)
    bullish_candles = len(candle_df[candle_df['candle_10am_2pm_direction'] == 'bullish'])
    
    print(f"\n10am-2pm Candle Direction:")
    print(f"  Total: {total_candles}")
    print(f"  Bullish: {bullish_candles} ({bullish_candles/total_candles*100:.1f}%)")
    print(f"  Bearish: {total_candles - bullish_candles} ({(total_candles - bullish_candles)/total_candles*100:.1f}%)")
    print(f"\n  ** BASELINE EXPECTATION: {bullish_candles/total_candles*100:.1f}% bullish **")

# ============================================================
# YEAR BY YEAR BREAKDOWN
# ============================================================
print("\n" + "="*100)
print("YEAR-BY-YEAR BREAKDOWN (Market Bias Over Time)")
print("="*100)

if 'Date' in df.columns:
    df['year'] = pd.to_datetime(df['Date']).dt.year
    year_df = df[df['ny_direction'].notna()].groupby('year')['ny_direction'].apply(
        lambda x: pd.Series({
            'total': len(x),
            'bullish': (x == 'bullish').sum(),
            'bullish_pct': (x == 'bullish').sum() / len(x) * 100
        })
    ).reset_index()
    
    print("\nYear    Total Days    Bullish    Bearish    Bullish %")
    print("-" * 100)
    for _, row in year_df.iterrows():
        year = row['year']
        total = int(row['total'])
        bullish = int(row['bullish'])
        bearish = total - bullish
        pct = row['bullish_pct']
        print(f"{year}       {total:3d}         {bullish:3d}        {bearish:3d}        {pct:5.1f}%")

# ============================================================
# KEY INSIGHTS
# ============================================================
print("\n" + "="*100)
print("KEY INSIGHTS FOR PATTERN EVALUATION")
print("="*100)

baseline = bullish_days/total_days*100

print(f"\n1. Overall market bias: {baseline:.1f}% bullish")
print(f"   → Any pattern showing <{baseline+5:.0f}% bullish has WEAK edge")
print(f"   → Look for patterns >70% (>{baseline+15:.0f}% above baseline) for strong edge")

print(f"\n2. True edge calculation:")
print(f"   → Edge = Pattern Confidence - Baseline ({baseline:.1f}%)")
print(f"   → Example: 75% pattern = {75-baseline:.1f}% true edge")

print(f"\n3. Why all top patterns are bullish:")
print(f"   → Baseline is {baseline:.1f}% bullish")
print(f"   → Bearish patterns need >{100-baseline:.1f}% to compete")
print(f"   → That's why you see no bearish patterns in top 100")

print(f"\n4. What to look for:")
print(f"   → Bullish patterns with >{baseline+20:.0f}% confidence (strong bull signal)")
print(f"   → Bearish patterns with >{100-baseline:.0f}% confidence (strong bear signal)")
print(f"   → High LIFT values (>1.3x) = true predictive power")

print("\n" + "="*100)

BASELINE STATISTICS - UNDERSTANDING MARKET BIAS

Analyzing 3126 trading days from 2013-2025

OVERALL BASELINE

NY Session Direction:
  Total days: 3126
  Bullish: 1708 (54.6%)
  Bearish: 1418 (45.4%)

  ** BASELINE EXPECTATION: 54.6% bullish **

BASELINE BY KEY FEATURES (What % bullish given each condition?)

NY ORB Direction (ny_orb_direction):
----------------------------------------------------------------------------------------------------
  bullish                       :  67.8% bullish (1114/1644 days)
  bearish                       :  40.2% bullish ( 594/1479 days)

NY ORB First Break (ny_orb_first_break):
----------------------------------------------------------------------------------------------------
  high                          :  70.1% bullish (1126/1606 days)
  low                           :  38.4% bullish ( 581/1513 days)

9:35 ORB Direction (orb_935_direction):
----------------------------------------------------------------------------------------------------
  

In [15]:
## SMART Pattern Discovery - Adaptive Feature Selection
## Only mines 3-feature patterns when they add real incremental value

import pandas as pd
import numpy as np
from itertools import combinations

# Load dataset
input_dir = r"C:\Users\tammm\NQ and ES analysis"
unified_df = pd.read_csv(f'{input_dir}\\ultimate_comprehensive_dataset.csv')

print("="*100)
print("SMART PATTERN DISCOVERY - ADAPTIVE COMPLEXITY")
print("="*100)
print(f"Loaded {len(unified_df)} trading days")
print()
print("STRATEGY:")
print("  1. Mine all 1-feature and 2-feature patterns")
print("  2. For 3-feature patterns, only keep if:")
print("     - Incremental edge from 3rd feature ≥ 4%")
print("     - Sample size ≥ 150 days")
print("     - Absolute confidence improvement ≥ 5%")
print("  3. Never mine 4+ features (diminishing returns)")
print()

# Features available at each entry time
FEATURES_945 = [
    'prev_day_direction', 'gap_type', 'gap_size', 'ny_open_vs_prev_day',
    'ny_open_vs_prev_week', 'day_of_week', 'ny_open_pivot_position',
    'overnight_highest_pivot', 'overnight_lowest_pivot', 'overnight_touched_pp',
    'overnight_touched_gz', 'overnight_touched_r1', 'overnight_touched_r2',
    'overnight_touched_r3', 'overnight_touched_s1', 'overnight_touched_s2',
    'overnight_touched_s3', 'overnight_touched_prev_day_high',
    'overnight_touched_prev_day_low', 'overnight_touched_prev_week_high',
    'overnight_touched_prev_week_low', 'overnight_direction', 'session_pattern',
    'asia_direction', 'london_direction', 'globex_6pm_direction',
    'asia_8pm_direction', 'midnight_direction', 'london_2am_direction',
    'hour_7am_direction', 'ny_futures_8am_direction', 'candle_4h_6pm_direction',
    'candle_4h_10pm_direction', 'candle_4h_2am_direction', 'globex_orb_direction',
    'asia_orb_direction', 'london_orb_direction', 'premarket_orb_direction',
    'range_6_9_direction', 'ny_open_vs_overnight', 'ny_orb_direction',
    'ny_orb_first_break', 'orb_935_direction', 'orb_935_first_break'
]

FEATURES_1030 = FEATURES_945 + [
    'ib_direction', 'candle_4h_6am_direction',
    'touched_prev_day_high_before_1030', 'touched_prev_day_low_before_1030',
    'touched_prev_week_high_before_1030', 'touched_prev_week_low_before_1030'
]

# Filter to available
features_945 = [f for f in FEATURES_945 if f in unified_df.columns and unified_df[f].notna().sum() > 50]
features_1030 = [f for f in FEATURES_1030 if f in unified_df.columns and unified_df[f].notna().sum() > 50]

# Thresholds
MIN_SUPPORT = 0.008
MIN_CONFIDENCE = 0.58
MIN_SAMPLE = 20
MIN_SAMPLE_3FEAT = 150
MIN_INCREMENTAL_3FEAT = 0.04  # 4% incremental improvement
MIN_ABSOLUTE_IMPROVE_3FEAT = 0.05  # 5% absolute improvement

def mine_smart_patterns(feature_list, feature_set_name, outcome_var='ny_direction'):
    """Smart pattern mining with adaptive complexity"""
    
    print(f"\n{'='*100}")
    print(f"MINING PATTERNS FOR {feature_set_name}")
    print(f"{'='*100}")
    print(f"Testing {len(feature_list)} features")
    print(f"Outcome: {outcome_var}")
    
    all_patterns = []
    
    # ========================================
    # PHASE 1: Mine ALL 1-feature patterns
    # ========================================
    print(f"\n  Phase 1: Mining 1-feature patterns...")
    
    for f1 in feature_list:
        valid_mask = unified_df[f1].notna() & unified_df[outcome_var].notna()
        subset = unified_df[valid_mask]
        
        if len(subset) < MIN_SAMPLE:
            continue
        
        grouped = subset.groupby([f1, outcome_var]).size()
        
        for (v1, out), count in grouped.items():
            support = count / len(subset)
            if support < MIN_SUPPORT:
                continue
            
            condition_mask = (subset[f1] == v1)
            condition_count = condition_mask.sum()
            
            if condition_count == 0:
                continue
            
            confidence = count / condition_count
            if confidence < MIN_CONFIDENCE:
                continue
            
            baseline_prob = (subset[outcome_var] == out).sum() / len(subset)
            lift = confidence / baseline_prob if baseline_prob > 0 else 0
            
            all_patterns.append({
                'feature_count': 1,
                'f1': f1, 'v1': v1,
                'outcome': out,
                'support': support,
                'confidence': confidence,
                'lift': lift,
                'count': count,
                'condition_count': condition_count,
                'incremental_edge': None,  # No incremental for 1-feature
                'passed_incremental_test': True
            })
    
    print(f"    Found {len([p for p in all_patterns if p['feature_count']==1])} 1-feature patterns")
    
    # ========================================
    # PHASE 2: Mine ALL 2-feature patterns
    # ========================================
    print(f"  Phase 2: Mining 2-feature patterns...")
    
    for f1, f2 in combinations(feature_list, 2):
        valid_mask = unified_df[f1].notna() & unified_df[f2].notna() & unified_df[outcome_var].notna()
        subset = unified_df[valid_mask]
        
        if len(subset) < MIN_SAMPLE:
            continue
        
        grouped = subset.groupby([f1, f2, outcome_var]).size()
        
        for (v1, v2, out), count in grouped.items():
            support = count / len(subset)
            if support < MIN_SUPPORT:
                continue
            
            condition_mask = (subset[f1] == v1) & (subset[f2] == v2)
            condition_count = condition_mask.sum()
            
            if condition_count == 0:
                continue
            
            confidence = count / condition_count
            if confidence < MIN_CONFIDENCE:
                continue
            
            baseline_prob = (subset[outcome_var] == out).sum() / len(subset)
            lift = confidence / baseline_prob if baseline_prob > 0 else 0
            
            all_patterns.append({
                'feature_count': 2,
                'f1': f1, 'v1': v1,
                'f2': f2, 'v2': v2,
                'outcome': out,
                'support': support,
                'confidence': confidence,
                'lift': lift,
                'count': count,
                'condition_count': condition_count,
                'incremental_edge': None,
                'passed_incremental_test': True
            })
    
    print(f"    Found {len([p for p in all_patterns if p['feature_count']==2])} 2-feature patterns")
    
    # ========================================
    # PHASE 3: Mine SELECTIVE 3-feature patterns
    # ========================================
    print(f"  Phase 3: Mining 3-feature patterns (with incremental filtering)...")
    
    checked_3feat = 0
    kept_3feat = 0
    rejected_3feat = 0
    
    for f1, f2, f3 in combinations(feature_list, 3):
        valid_mask = (unified_df[f1].notna() & unified_df[f2].notna() & 
                      unified_df[f3].notna() & unified_df[outcome_var].notna())
        subset = unified_df[valid_mask]
        
        if len(subset) < MIN_SAMPLE:
            continue
        
        grouped = subset.groupby([f1, f2, f3, outcome_var]).size()
        
        for (v1, v2, v3, out), count in grouped.items():
            checked_3feat += 1
            
            support = count / len(subset)
            if support < MIN_SUPPORT:
                continue
            
            condition_mask = (subset[f1] == v1) & (subset[f2] == v2) & (subset[f3] == v3)
            condition_count = condition_mask.sum()
            
            if condition_count < MIN_SAMPLE_3FEAT:
                rejected_3feat += 1
                continue
            
            confidence = count / condition_count
            if confidence < MIN_CONFIDENCE:
                continue
            
            # Calculate baseline from 2-feature pattern
            condition_mask_2feat = (subset[f1] == v1) & (subset[f2] == v2)
            baseline_2feat_subset = subset[condition_mask_2feat]
            
            if len(baseline_2feat_subset) == 0:
                rejected_3feat += 1
                continue
            
            baseline_2feat_conf = len(baseline_2feat_subset[baseline_2feat_subset[outcome_var] == out]) / len(baseline_2feat_subset)
            incremental_edge = confidence - baseline_2feat_conf
            
            # INCREMENTAL TEST
            passed_incremental = (
                abs(incremental_edge) >= MIN_INCREMENTAL_3FEAT and
                abs(confidence - 0.5) >= (abs(baseline_2feat_conf - 0.5) + MIN_ABSOLUTE_IMPROVE_3FEAT)
            )
            
            if not passed_incremental:
                rejected_3feat += 1
                continue
            
            baseline_prob = (subset[outcome_var] == out).sum() / len(subset)
            lift = confidence / baseline_prob if baseline_prob > 0 else 0
            
            kept_3feat += 1
            all_patterns.append({
                'feature_count': 3,
                'f1': f1, 'v1': v1,
                'f2': f2, 'v2': v2,
                'f3': f3, 'v3': v3,
                'outcome': out,
                'support': support,
                'confidence': confidence,
                'lift': lift,
                'count': count,
                'condition_count': condition_count,
                'incremental_edge': incremental_edge,
                'passed_incremental_test': True
            })
    
    print(f"    Checked {checked_3feat} 3-feature combinations")
    print(f"    Kept {kept_3feat} patterns (passed incremental test)")
    print(f"    Rejected {rejected_3feat} patterns (failed incremental test)")
    
    print(f"\n  Total patterns found: {len(all_patterns)}")
    
    return all_patterns

# Mine patterns for both entry times
print("\n" + "="*100)
print("STARTING PATTERN MINING")
print("="*100)

rules_945 = mine_smart_patterns(features_945, "9:45AM ENTRY", outcome_var='ny_direction')
rules_1030 = mine_smart_patterns(features_1030, "10:30AM ENTRY", outcome_var='ny_direction')

# Convert to dataframes
df_945 = pd.DataFrame(rules_945) if rules_945 else pd.DataFrame()
df_1030 = pd.DataFrame(rules_1030) if rules_1030 else pd.DataFrame()

# Calculate actionability score
def calc_score(row):
    edge = abs(row['confidence'] - 0.5) * 100
    confidence_score = min(edge * 2, 100)
    sample = row['condition_count']
    if sample < 50:
        freq_score = (sample / 50) * 50
    else:
        freq_score = min(50 + (np.log2(sample / 50) * 25), 100)
    return round((confidence_score * 0.6) + (freq_score * 0.4), 1)

if not df_945.empty:
    df_945['score'] = df_945.apply(calc_score, axis=1)
    df_945['edge_pct'] = abs(df_945['confidence'] - 0.5) * 100
    df_945 = df_945.sort_values('score', ascending=False)

if not df_1030.empty:
    df_1030['score'] = df_1030.apply(calc_score, axis=1)
    df_1030['edge_pct'] = abs(df_1030['confidence'] - 0.5) * 100
    df_1030 = df_1030.sort_values('score', ascending=False)

# Save results
print("\n" + "="*100)
print("SAVING RESULTS")
print("="*100)

if not df_945.empty:
    df_945.to_csv(f'{input_dir}\\smart_rules_945am.csv', index=False)
    print(f"✓ Saved: smart_rules_945am.csv ({len(df_945)} patterns)")

if not df_1030.empty:
    df_1030.to_csv(f'{input_dir}\\smart_rules_1030am.csv', index=False)
    print(f"✓ Saved: smart_rules_1030am.csv ({len(df_1030)} patterns)")

# Summary by feature count
print("\n" + "="*100)
print("PATTERN DISTRIBUTION BY COMPLEXITY")
print("="*100)

for entry_name, df in [("9:45AM", df_945), ("10:30AM", df_1030)]:
    if df.empty:
        continue
    
    print(f"\n{entry_name}:")
    print("-"*100)
    
    for feat_count in [1, 2, 3]:
        patterns = df[df['feature_count'] == feat_count]
        
        if len(patterns) == 0:
            continue
        
        bull = patterns[patterns['outcome'] == 'bullish']
        bear = patterns[patterns['outcome'] == 'bearish']
        
        print(f"\n  {feat_count}-Feature Patterns:")
        print(f"    Total: {len(patterns):,} patterns")
        print(f"    Bullish: {len(bull):,} | Avg conf: {bull['confidence'].mean()*100:.1f}% | "
              f"Avg sample: {bull['condition_count'].mean():.0f} days" if len(bull) > 0 else "    Bullish: 0")
        print(f"    Bearish: {len(bear):,} | Avg conf: {bear['confidence'].mean()*100:.1f}% | "
              f"Avg sample: {bear['condition_count'].mean():.0f} days" if len(bear) > 0 else "    Bearish: 0")
        
        if feat_count == 3 and len(patterns) > 0:
            avg_incr = patterns['incremental_edge'].mean()
            print(f"    Avg incremental edge from 3rd feature: {avg_incr*100:+.1f}%")

# Key insights
print("\n\n" + "="*100)
print("KEY INSIGHTS")
print("="*100)

print("\n1. PATTERN QUALITY:")
print("   - 1-feature: Simple, high frequency, moderate edge")
print("   - 2-feature: Best balance of edge and frequency")
print("   - 3-feature: ONLY kept when 3rd feature adds real value (4%+ incremental)")

print("\n2. COMPARED TO ORIGINAL MINER:")
if not df_945.empty and not df_1030.empty:
    print(f"   Original 9:45am patterns: 37,884 (many useless 3-4 feature patterns)")
    print(f"   Smart 9:45am patterns: {len(df_945):,} (only quality patterns)")
    print(f"   Original 10:30am patterns: 52,921")
    print(f"   Smart 10:30am patterns: {len(df_1030):,}")

print("\n3. EXPECTED RESULTS:")
print("   - Mostly 1-2 feature patterns (high frequency)")
print("   - Few 3-feature patterns (only when truly incremental)")
print("   - Bearish patterns more likely to have 3 features (need refinement)")
print("   - Bullish patterns mostly 1-2 features (already strong)")

print("\n4. USAGE:")
print("   - Trade 1-2 feature patterns for daily setups")
print("   - Use 3-feature patterns for high-confidence rare setups")
print("   - All patterns guaranteed to have incremental value")

print("\n" + "="*100)
print("ANALYSIS COMPLETE")
print("="*100)

SMART PATTERN DISCOVERY - ADAPTIVE COMPLEXITY
Loaded 3126 trading days

STRATEGY:
  1. Mine all 1-feature and 2-feature patterns
  2. For 3-feature patterns, only keep if:
     - Incremental edge from 3rd feature ≥ 4%
     - Sample size ≥ 150 days
     - Absolute confidence improvement ≥ 5%
  3. Never mine 4+ features (diminishing returns)


STARTING PATTERN MINING

MINING PATTERNS FOR 9:45AM ENTRY
Testing 44 features
Outcome: ny_direction

  Phase 1: Mining 1-feature patterns...
    Found 14 1-feature patterns
  Phase 2: Mining 2-feature patterns...
    Found 1133 2-feature patterns
  Phase 3: Mining 3-feature patterns (with incremental filtering)...
    Checked 340337 3-feature combinations
    Kept 14447 patterns (passed incremental test)
    Rejected 100421 patterns (failed incremental test)

  Total patterns found: 15594

MINING PATTERNS FOR 10:30AM ENTRY
Testing 50 features
Outcome: ny_direction

  Phase 1: Mining 1-feature patterns...
    Found 19 1-feature patterns
  Phase 2: M

In [1]:
import pandas as pd
import numpy as np

input_dir = r"C:\Users\tammm\NQ and ES analysis"

# Load smart patterns for NY session
print("Loading SMART patterns for NY session (9:30-4pm)...")
df_945 = pd.read_csv(f'{input_dir}\\smart_rules_945am.csv')
df_1030 = pd.read_csv(f'{input_dir}\\smart_rules_1030am.csv')

BASELINE_BULLISH = 0.546
BASELINE_BEARISH = 0.454

# Calculate scores
def calc_score(row):
    edge = abs(row['confidence'] - 0.5) * 100
    confidence_score = min(edge * 2, 100)
    sample = row['condition_count']
    if sample < 50:
        freq_score = (sample / 50) * 50
    else:
        freq_score = min(50 + (np.log2(sample / 50) * 25), 100)
    return round((confidence_score * 0.6) + (freq_score * 0.4), 1)

df_945['score'] = df_945.apply(calc_score, axis=1)
df_1030['score'] = df_1030.apply(calc_score, axis=1)

print("\n" + "="*130)
print("SMART PATTERNS - ORGANIZED BY COMPLEXITY (NY SESSION)")
print("="*130)
print("Shows: Top patterns by feature count (1, 2, 3 features)")
print("Outcome: NY session direction (9:30-4pm)")
print()

def display_patterns_by_complexity(df, entry_name):
    print("\n" + "="*130)
    print(f"{entry_name} PATTERNS")
    print("="*130)
    
    for feat_count in [1, 2, 3]:
        patterns = df[df['feature_count'] == feat_count].sort_values('score', ascending=False)
        
        if len(patterns) == 0:
            continue
        
        bullish = patterns[patterns['outcome'] == 'bullish']
        bearish = patterns[patterns['outcome'] == 'bearish']
        
        print(f"\n{'-'*130}")
        print(f"{feat_count}-FEATURE PATTERNS")
        print(f"{'-'*130}")
        print(f"Total: {len(patterns):,} | Bullish: {len(bullish):,} | Bearish: {len(bearish):,}")
        
        # Show top 10 bullish
        print(f"\nTOP 10 BULLISH ({feat_count}-feature):")
        print(f"{'#':<3} {'Score':<6} {'Conf':<7} {'Edge':<7} {'Days':<6} {'Conditions':<80}")
        print("-"*130)
        
        for i, (idx, row) in enumerate(bullish.head(10).iterrows(), 1):
            # Build conditions
            cond = f"{row['f1']}={row['v1']}"
            if feat_count >= 2:
                cond += f", {row['f2']}={row['v2']}"
            if feat_count >= 3:
                cond += f", {row['f3']}={row['v3']}"
            
            cond = cond[:80]
            true_edge = (row['confidence'] - BASELINE_BULLISH) * 100
            
            print(f"{i:<3} {row['score']:<6.1f} {row['confidence']*100:<6.1f}% {true_edge:+6.1f}% {row['condition_count']:<6} {cond}")
        
        # Show top 10 bearish
        print(f"\nTOP 10 BEARISH ({feat_count}-feature):")
        print(f"{'#':<3} {'Score':<6} {'Conf':<7} {'Edge':<7} {'Days':<6} {'Conditions':<80}")
        print("-"*130)
        
        if len(bearish) > 0:
            for i, (idx, row) in enumerate(bearish.head(10).iterrows(), 1):
                # Build conditions
                cond = f"{row['f1']}={row['v1']}"
                if feat_count >= 2:
                    cond += f", {row['f2']}={row['v2']}"
                if feat_count >= 3:
                    cond += f", {row['f3']}={row['v3']}"
                
                cond = cond[:80]
                true_edge = (row['confidence'] - BASELINE_BEARISH) * 100
                
                print(f"{i:<3} {row['score']:<6.1f} {row['confidence']*100:<6.1f}% {true_edge:+6.1f}% {row['condition_count']:<6} {cond}")
        else:
            print("  No bearish patterns")

# Display both entry times
display_patterns_by_complexity(df_945, "9:45AM ENTRY")
display_patterns_by_complexity(df_1030, "10:30AM ENTRY")

# Summary comparison
print("\n\n" + "="*130)
print("SUMMARY - PATTERN QUALITY BY COMPLEXITY")
print("="*130)

for entry_name, df in [("9:45AM", df_945), ("10:30AM", df_1030)]:
    if df.empty:
        continue
    
    print(f"\n{entry_name}:")
    print("-"*130)
    
    for feat_count in [1, 2, 3]:
        patterns = df[df['feature_count'] == feat_count]
        
        if len(patterns) == 0:
            continue
        
        bullish = patterns[patterns['outcome'] == 'bullish']
        bearish = patterns[patterns['outcome'] == 'bearish']
        
        print(f"\n  {feat_count}-Feature Patterns:")
        print(f"    Total: {len(patterns):,}")
        
        if len(bullish) > 0:
            print(f"    Bullish: {len(bullish):,} patterns")
            print(f"      Best confidence: {bullish['confidence'].max()*100:.1f}%")
            print(f"      Best true edge: {(bullish['confidence'].max() - BASELINE_BULLISH)*100:+.1f}%")
            print(f"      Avg sample: {bullish['condition_count'].mean():.0f} days")
        
        if len(bearish) > 0:
            print(f"    Bearish: {len(bearish):,} patterns")
            print(f"      Best confidence: {bearish['confidence'].max()*100:.1f}%")
            print(f"      Best true edge: {(bearish['confidence'].max() - BASELINE_BEARISH)*100:+.1f}%")
            print(f"      Avg sample: {bearish['condition_count'].mean():.0f} days")
        
        if feat_count == 3 and len(patterns) > 0:
            avg_incr = patterns['incremental_edge'].mean() * 100
            print(f"    Avg incremental from 3rd feature: {avg_incr:+.1f}%")

# Key recommendations
print("\n\n" + "="*130)
print("TRADING RECOMMENDATIONS - NY SESSION")
print("="*130)

print("\n1. DAILY TRADING (High Frequency):")
print("   → Use 1-2 feature patterns")
print("   → Examples:")

# Get best 1-2 feature patterns
simple_945_bull = df_945[(df_945['feature_count'] <= 2) & (df_945['outcome'] == 'bullish')].sort_values('score', ascending=False).head(3)
simple_945_bear = df_945[(df_945['feature_count'] <= 2) & (df_945['outcome'] == 'bearish')].sort_values('score', ascending=False).head(3)

print("\n   9:45am Bullish:")
for i, (idx, row) in enumerate(simple_945_bull.iterrows(), 1):
    cond = f"{row['f1']}={row['v1']}"
    if row['feature_count'] >= 2:
        cond += f", {row['f2']}={row['v2']}"
    print(f"     {i}. {cond[:70]} → {row['confidence']*100:.1f}% ({row['condition_count']} days)")

print("\n   9:45am Bearish:")
for i, (idx, row) in enumerate(simple_945_bear.iterrows(), 1):
    cond = f"{row['f1']}={row['v1']}"
    if row['feature_count'] >= 2:
        cond += f", {row['f2']}={row['v2']}"
    print(f"     {i}. {cond[:70]} → {row['confidence']*100:.1f}% ({row['condition_count']} days)")

print("\n2. HIGH-CONFIDENCE SETUPS (Lower Frequency):")
print("   → Use 3-feature patterns")
print("   → Only when all 3 conditions align")
print("   → Examples:")

# Get best 3-feature patterns
refined_945_bull = df_945[(df_945['feature_count'] == 3) & (df_945['outcome'] == 'bullish')].sort_values('score', ascending=False).head(3)
refined_945_bear = df_945[(df_945['feature_count'] == 3) & (df_945['outcome'] == 'bearish')].sort_values('score', ascending=False).head(3)

print("\n   9:45am Bullish:")
for i, (idx, row) in enumerate(refined_945_bull.iterrows(), 1):
    cond = f"{row['f1']}={row['v1']}, {row['f2']}={row['v2']}, {row['f3']}={row['v3']}"
    print(f"     {i}. {cond[:70]} → {row['confidence']*100:.1f}% ({row['condition_count']} days)")

print("\n   9:45am Bearish:")
for i, (idx, row) in enumerate(refined_945_bear.iterrows(), 1):
    cond = f"{row['f1']}={row['v1']}, {row['f2']}={row['v2']}, {row['f3']}={row['v3']}"
    print(f"     {i}. {cond[:70]} → {row['confidence']*100:.1f}% ({row['condition_count']} days)")

print("\n3. INCREMENTAL VALUE:")
print("   → All 3-feature patterns add ≥4% edge from 3rd feature")
print("   → Average incremental: 12.6% (9:45am) / 14.0% (10:30am)")
print("   → This means complexity IS worth it for these patterns")

print("\n" + "="*130)

Loading SMART patterns for NY session (9:30-4pm)...

SMART PATTERNS - ORGANIZED BY COMPLEXITY (NY SESSION)
Shows: Top patterns by feature count (1, 2, 3 features)
Outcome: NY session direction (9:30-4pm)


9:45AM ENTRY PATTERNS

----------------------------------------------------------------------------------------------------------------------------------
1-FEATURE PATTERNS
----------------------------------------------------------------------------------------------------------------------------------
Total: 14 | Bullish: 11 | Bearish: 3

TOP 10 BULLISH (1-feature):
#   Score  Conf    Edge    Days   Conditions                                                                      
----------------------------------------------------------------------------------------------------------------------------------
1   64.1   70.1  %  +15.5% 1606   ny_orb_first_break=high
2   61.3   67.8  %  +13.2% 1644   ny_orb_direction=bullish
3   58.6   65.5  %  +10.9% 1643   orb_935_first_break=high
4 

In [2]:
import pandas as pd
import numpy as np

input_dir = r"C:\Users\tammm\NQ and ES analysis"

# Load dataset
df = pd.read_csv(f'{input_dir}\\ultimate_comprehensive_dataset.csv')

print("="*130)
print("INCREMENTAL VALUE CALCULATOR - NY SESSION")
print("="*130)
print("Shows: How much does each additional feature improve prediction?")
print("Outcome: ny_direction (9:30-4pm)")
print()

BASELINE = 0.546

# Test specific feature combinations
test_combos = [
    # ORB-based progressions
    {
        'name': 'ORB Progression (Bullish)',
        'features': [
            ('ny_orb_direction', 'bullish'),
            ('ny_orb_first_break', 'high'),
            ('overnight_touched_r1', True),
            ('day_of_week', 'Monday')
        ]
    },
    # IB-based progressions
    {
        'name': 'IB Progression (Bullish)',
        'features': [
            ('ib_direction', 'bullish'),
            ('ny_orb_first_break', 'high'),
            ('overnight_touched_r1', True),
            ('session_pattern', 'london_partial_up')
        ]
    },
    # Bearish ORB progression
    {
        'name': 'ORB Progression (Bearish)',
        'features': [
            ('ny_orb_direction', 'bearish'),
            ('ny_orb_first_break', 'low'),
            ('overnight_touched_r1', True),
            ('asia_direction', 'bullish')
        ]
    },
    # Bearish IB progression
    {
        'name': 'IB Progression (Bearish)',
        'features': [
            ('ib_direction', 'bearish'),
            ('ny_orb_first_break', 'low'),
            ('overnight_touched_r1', True),
            ('gap_type', 'gap_up')
        ]
    }
]

def calculate_incremental(features_list, df):
    """Calculate incremental improvement by adding features one by one"""
    
    results = []
    cumulative_mask = pd.Series([True] * len(df))
    
    for i, (feature, value) in enumerate(features_list):
        if feature not in df.columns:
            continue
        
        # Add this feature to the cumulative filter
        cumulative_mask = cumulative_mask & (df[feature] == value)
        
        # Get subset
        subset = df[cumulative_mask & df['ny_direction'].notna()]
        
        if len(subset) < 10:
            results.append({
                'step': i + 1,
                'feature': feature,
                'value': value,
                'sample_days': len(subset),
                'bullish_pct': None,
                'edge': None,
                'incremental': None,
                'status': 'Too few samples'
            })
            continue
        
        bullish_count = len(subset[subset['ny_direction'] == 'bullish'])
        bullish_pct = bullish_count / len(subset)
        
        # Calculate metrics
        edge = abs(bullish_pct - 0.5) * 100
        
        # Calculate incremental from previous step
        if i == 0:
            incremental = (bullish_pct - BASELINE) * 100
            comparison_base = BASELINE * 100
        else:
            prev_pct = results[i-1]['bullish_pct']
            if prev_pct is not None:
                incremental = (bullish_pct - prev_pct) * 100
                comparison_base = prev_pct * 100
            else:
                incremental = None
                comparison_base = None
        
        results.append({
            'step': i + 1,
            'feature': feature,
            'value': value,
            'sample_days': len(subset),
            'bullish_count': bullish_count,
            'bullish_pct': bullish_pct,
            'edge': edge,
            'incremental': incremental,
            'comparison_base': comparison_base,
            'status': 'OK'
        })
    
    return results

# Analyze each combo
for combo in test_combos:
    print("\n" + "="*130)
    print(f"{combo['name'].upper()}")
    print("="*130)
    
    results = calculate_incremental(combo['features'], df)
    
    print("\nStep | Feature + Value                                  | Days  | Bullish | Edge  | +Incr  | Worth It?")
    print("-"*130)
    
    for r in results:
        if r['status'] != 'OK':
            print(f"{r['step']:4d} | {r['feature']:30s} = {str(r['value']):15s} | Too few samples")
            continue
        
        feature_str = f"{r['feature']} = {r['value']}"
        
        # Determine if incremental is worth it
        worth_it = ""
        if r['incremental'] is not None:
            if r['step'] == 1:
                worth_it = "✓ Base" if abs(r['incremental']) > 10 else "⚠ Weak base"
            else:
                worth_it = "✓ Yes" if abs(r['incremental']) >= 5 else "⚠ Maybe" if abs(r['incremental']) >= 2 else "✗ No"
        
        incr_str = f"{r['incremental']:+.1f}%" if r['incremental'] is not None else "N/A"
        
        print(f"{r['step']:4d} | {feature_str:48s} | {r['sample_days']:5d} | "
              f"{r['bullish_pct']*100:6.1f}% | {r['edge']:5.1f}% | {incr_str:7s} | {worth_it}")
    
    # Summary for this combo
    print("\nSUMMARY:")
    valid_results = [r for r in results if r['status'] == 'OK']
    if len(valid_results) > 0:
        first = valid_results[0]
        last = valid_results[-1]
        total_improvement = (last['bullish_pct'] - first['bullish_pct']) * 100 if len(valid_results) > 1 else 0
        
        print(f"  Start: {first['feature']} alone = {first['bullish_pct']*100:.1f}% ({first['sample_days']} days)")
        print(f"  End: All {len(valid_results)} features = {last['bullish_pct']*100:.1f}% ({last['sample_days']} days)")
        print(f"  Total improvement: {total_improvement:+.1f}%")
        print(f"  Cost: Sample reduced from {first['sample_days']} to {last['sample_days']} days ({(1-last['sample_days']/first['sample_days'])*100:.0f}% fewer)")

# General analysis across top patterns
print("\n\n")
print("="*130)
print("PATTERN COMPLEXITY ANALYSIS - ARE 3-4 FEATURE PATTERNS WORTH IT?")
print("="*130)

# Load actual patterns
df_945 = pd.read_csv(f'{input_dir}\\smart_rules_945am.csv')
df_1030 = pd.read_csv(f'{input_dir}\\smart_rules_1030am.csv')

print("\n9:45AM PATTERNS:")
print("-"*130)

# Group by feature count
for feature_count in [2, 3, 4]:
    patterns = df_945[df_945['feature_count'] == feature_count]
    
    if len(patterns) == 0:
        continue
    
    # Get patterns with good sample sizes
    good_patterns = patterns[patterns['condition_count'] >= 100]
    
    if len(good_patterns) == 0:
        continue
    
    avg_conf = good_patterns['confidence'].mean()
    max_conf = good_patterns['confidence'].max()
    avg_sample = good_patterns['condition_count'].mean()
    
    print(f"\n{feature_count}-Feature Patterns (100+ days):")
    print(f"  Count: {len(good_patterns):,}")
    print(f"  Avg confidence: {avg_conf*100:.1f}%")
    print(f"  Max confidence: {max_conf*100:.1f}%")
    print(f"  Avg sample: {avg_sample:.0f} days")

print("\n\n10:30AM PATTERNS:")
print("-"*130)

for feature_count in [2, 3, 4]:
    patterns = df_1030[df_1030['feature_count'] == feature_count]
    
    if len(patterns) == 0:
        continue
    
    good_patterns = patterns[patterns['condition_count'] >= 100]
    
    if len(good_patterns) == 0:
        continue
    
    avg_conf = good_patterns['confidence'].mean()
    max_conf = good_patterns['confidence'].max()
    avg_sample = good_patterns['condition_count'].mean()
    
    print(f"\n{feature_count}-Feature Patterns (100+ days):")
    print(f"  Count: {len(good_patterns):,}")
    print(f"  Avg confidence: {avg_conf*100:.1f}%")
    print(f"  Max confidence: {max_conf*100:.1f}%")
    print(f"  Avg sample: {avg_sample:.0f} days")

# Final recommendations
print("\n\n")
print("="*130)
print("RECOMMENDATIONS")
print("="*130)

print("\n1. INCREMENTAL VALUE THRESHOLD:")
print("   ✓ Feature adds ≥5% → Worth including")
print("   ⚠ Feature adds 2-5% → Maybe worth it")
print("   ✗ Feature adds <2% → Just noise, skip it")

print("\n2. COMPLEXITY vs FREQUENCY TRADEOFF:")
print("   - 1 feature: High frequency, moderate edge")
print("   - 2 features: Good balance (sweet spot)")
print("   - 3 features: Better edge, lower frequency")
print("   - 4+ features: Diminishing returns")

print("\n3. SIMPLE > COMPLEX:")
print("   If ORB alone = 70% and ORB + 2 features = 77%:")
print("   - You gain 7% edge")
print("   - But lose 80% of opportunities")
print("   - Trade-off may not be worth it!")

print("\n4. SUGGESTED APPROACH:")
print("   → Trade 1-2 feature patterns for high frequency")
print("   → Use 3+ features only when incremental adds 5%+")
print("   → Never go beyond 3 features unless sample stays >100 days")

print("\n" + "="*130)

INCREMENTAL VALUE CALCULATOR - NY SESSION
Shows: How much does each additional feature improve prediction?
Outcome: ny_direction (9:30-4pm)


ORB PROGRESSION (BULLISH)

Step | Feature + Value                                  | Days  | Bullish | Edge  | +Incr  | Worth It?
----------------------------------------------------------------------------------------------------------------------------------
   1 | ny_orb_direction = bullish                       |  1644 |   67.8% |  17.8% | +13.2%  | ✓ Base
   2 | ny_orb_first_break = high                        |  1209 |   74.4% |  24.4% | +6.6%   | ✓ Yes
   3 | overnight_touched_r1 = True                      |   239 |   77.0% |  27.0% | +2.6%   | ⚠ Maybe
   4 | day_of_week                    = Monday          | Too few samples

SUMMARY:
  Start: ny_orb_direction alone = 67.8% (1644 days)
  End: All 3 features = 77.0% (239 days)
  Total improvement: +9.2%
  Cost: Sample reduced from 1644 to 239 days (85% fewer)

IB PROGRESSION (BULLISH)

Ste

In [3]:
## Top 50 Bullish Patterns - NY Session (9:30-4pm)

import pandas as pd
import numpy as np

input_dir = r"C:\Users\tammm\NQ and ES analysis"

# Load pattern files
print("Loading patterns for NY session...")
df_945 = pd.read_csv(f'{input_dir}\\smart_rules_945am.csv')
df_1030 = pd.read_csv(f'{input_dir}\\smart_rules_1030am.csv')

# Baseline from previous analysis
BASELINE_BULLISH = 0.546
BASELINE_BEARISH = 0.454

print("\n" + "="*130)
print("TRADEABLE BULLISH PATTERNS - NY SESSION (9:30-4PM)")
print("="*130)
print("Minimum sample size: 50 days")
print("Minimum confidence: 65%")
print("Outcome: NY session direction (9:30-4pm)")
print()

# Filter criteria
MIN_SAMPLE = 50
MIN_CONFIDENCE = 0.65

# Calculate actionability score
def calc_score(row):
    edge = abs(row['confidence'] - 0.5) * 100
    confidence_score = min(edge * 2, 100)
    sample = row['condition_count']
    if sample < 50:
        freq_score = (sample / 50) * 50
    else:
        freq_score = min(50 + (np.log2(sample / 50) * 25), 100)
    return round((confidence_score * 0.6) + (freq_score * 0.4), 1)

df_945['score'] = df_945.apply(calc_score, axis=1)
df_1030['score'] = df_1030.apply(calc_score, axis=1)

# Calculate true edge
df_945['edge_pct'] = abs(df_945['confidence'] - 0.5) * 100
df_1030['edge_pct'] = abs(df_1030['confidence'] - 0.5) * 100

# Filter for tradeable bullish patterns
bullish_945 = df_945[
    (df_945['outcome'] == 'bullish') & 
    (df_945['condition_count'] >= MIN_SAMPLE) &
    (df_945['confidence'] >= MIN_CONFIDENCE)
].sort_values('score', ascending=False)

bullish_1030 = df_1030[
    (df_1030['outcome'] == 'bullish') & 
    (df_1030['condition_count'] >= MIN_SAMPLE) &
    (df_1030['confidence'] >= MIN_CONFIDENCE)
].sort_values('score', ascending=False)

# Display function
def display_patterns(df, title, n=50):
    print("\n" + "="*130)
    print(f"{title}")
    print("="*130)
    
    if len(df) == 0:
        print("No tradeable bullish patterns found with these criteria")
        return
    
    print(f"Found {len(df)} tradeable bullish patterns\n")
    
    for i, (idx, row) in enumerate(df.head(n).iterrows()):
        if i >= n:
            break
            
        # Build condition string
        cond = f"{row['f1']}={row['v1']}"
        if pd.notna(row.get('f2')):
            cond += f", {row['f2']}={row['v2']}"
        if pd.notna(row.get('f3')):
            cond += f", {row['f3']}={row['v3']}"
        
        # Calculate vs baseline
        true_edge = (row['confidence'] - BASELINE_BULLISH) * 100
        
        print(f"\n#{i+1} | Score: {row['score']:.1f} | Conf: {row['confidence']*100:.1f}% | "
              f"Edge: {row['edge_pct']:.1f}% | True Edge: {true_edge:+.1f}%")
        print(f"     Sample: {row['condition_count']} days | Lift: {row['lift']:.2f}x")
        print(f"     {cond}")
        print(f"     → BULLISH NY: {row['count']}/{row['condition_count']} days "
              f"({row['confidence']*100:.1f}% accuracy)")

# Display results
display_patterns(bullish_945, "TRADEABLE BULLISH PATTERNS - 9:45AM ENTRY", 50)
display_patterns(bullish_1030, "TRADEABLE BULLISH PATTERNS - 10:30AM ENTRY", 50)

# Summary statistics
print("\n" + "="*130)
print("SUMMARY - TRADEABLE BULLISH PATTERNS")
print("="*130)

print("\n9:45AM ENTRY:")
print(f"  Total bullish patterns: {len(df_945[df_945['outcome'] == 'bullish']):,}")
print(f"  Tradeable (50+ days, 65%+ conf): {len(bullish_945):,}")
if len(bullish_945) > 0:
    print(f"  Best confidence: {bullish_945['confidence'].max()*100:.1f}%")
    print(f"  Best score: {bullish_945['score'].max():.1f}")
    print(f"  Average confidence: {bullish_945['confidence'].mean()*100:.1f}%")
    print(f"  Average sample size: {bullish_945['condition_count'].mean():.0f} days")

print("\n10:30AM ENTRY:")
print(f"  Total bullish patterns: {len(df_1030[df_1030['outcome'] == 'bullish']):,}")
print(f"  Tradeable (50+ days, 65%+ conf): {len(bullish_1030):,}")
if len(bullish_1030) > 0:
    print(f"  Best confidence: {bullish_1030['confidence'].max()*100:.1f}%")
    print(f"  Best score: {bullish_1030['score'].max():.1f}")
    print(f"  Average confidence: {bullish_1030['confidence'].mean()*100:.1f}%")
    print(f"  Average sample size: {bullish_1030['condition_count'].mean():.0f} days")

# Compare to bearish
bearish_945 = df_945[
    (df_945['outcome'] == 'bearish') & 
    (df_945['condition_count'] >= MIN_SAMPLE) &
    (df_945['confidence'] >= MIN_CONFIDENCE)
]

bearish_1030 = df_1030[
    (df_1030['outcome'] == 'bearish') & 
    (df_1030['condition_count'] >= MIN_SAMPLE) &
    (df_1030['confidence'] >= MIN_CONFIDENCE)
]

print("\n" + "="*130)
print("COMPARISON: BULLISH vs BEARISH (Tradeable Patterns Only)")
print("="*130)

print("\n9:45AM:")
print(f"  Tradeable bullish: {len(bullish_945):,} patterns | Best conf: {bullish_945['confidence'].max()*100:.1f}%" if len(bullish_945) > 0 else "  Tradeable bullish: 0 patterns")
print(f"  Tradeable bearish: {len(bearish_945):,} patterns | Best conf: {bearish_945['confidence'].max()*100:.1f}%" if len(bearish_945) > 0 else "  Tradeable bearish: 0 patterns")

print("\n10:30AM:")
print(f"  Tradeable bullish: {len(bullish_1030):,} patterns | Best conf: {bullish_1030['confidence'].max()*100:.1f}%" if len(bullish_1030) > 0 else "  Tradeable bullish: 0 patterns")
print(f"  Tradeable bearish: {len(bearish_1030):,} patterns | Best conf: {bearish_1030['confidence'].max()*100:.1f}%" if len(bearish_1030) > 0 else "  Tradeable bearish: 0 patterns")

# Export tradeable patterns
if len(bullish_945) > 0:
    bullish_945.head(50).to_csv(f'{input_dir}\\tradeable_bullish_945am.csv', index=False)
    print(f"\n✓ Exported: tradeable_bullish_945am.csv ({len(bullish_945.head(50))} patterns)")

if len(bullish_1030) > 0:
    bullish_1030.head(50).to_csv(f'{input_dir}\\tradeable_bullish_1030am.csv', index=False)
    print(f"✓ Exported: tradeable_bullish_1030am.csv ({len(bullish_1030.head(50))} patterns)")

print("\n" + "="*130)
print("KEY INSIGHTS")
print("="*130)
print("\n1. Bullish patterns are COMMON because:")
print("   - Market baseline is 54.6% bullish (bull market period)")
print("   - Strong bullish setups occur frequently")
print("   - Most bullish patterns have large sample sizes (> 100 days)")

print("\n2. What makes a good bullish pattern:")
print("   - >70% bullish = 15.4% edge over baseline")
print("   - 50+ occurrences = tradeable frequency")
print("   - High lift (>1.3x) = true predictive power")

print("\n3. Expected baseline for common bullish setups:")
print("   - IB bullish + ORB high: 77.8% bullish (23.2% edge)")
print("   - Touched R1 + IB bullish: 80.1% bullish (25.5% edge)")
print("   - ORB bullish + breaks high: 74.4% bullish (19.8% edge)")

print("\n4. True edge calculation:")
print("   - Pattern at 75% bullish = 20.4% edge over 54.6% baseline")
print("   - Pattern at 80% bullish = 25.4% edge over 54.6% baseline")
print("   - Pattern at 85% bullish = 30.4% edge over 54.6% baseline")

print("\n" + "="*130)

Loading patterns for NY session...

TRADEABLE BULLISH PATTERNS - NY SESSION (9:30-4PM)
Minimum sample size: 50 days
Minimum confidence: 65%
Outcome: NY session direction (9:30-4pm)


TRADEABLE BULLISH PATTERNS - 9:45AM ENTRY
Found 6817 tradeable bullish patterns


#1 | Score: 74.0 | Conf: 80.6% | Edge: 30.6% | True Edge: +26.0%
     Sample: 165 days | Lift: 1.47x
     day_of_week=Friday, ny_orb_first_break=high, orb_935_direction=bullish
     → BULLISH NY: 133/165 days (80.6% accuracy)

#2 | Score: 74.0 | Conf: 78.3% | Edge: 28.3% | True Edge: +23.7%
     Sample: 212 days | Lift: 1.43x
     day_of_week=Monday, overnight_direction=bullish, ny_orb_first_break=high
     → BULLISH NY: 166/212 days (78.3% accuracy)

#3 | Score: 73.8 | Conf: 78.2% | Edge: 28.2% | True Edge: +23.6%
     Sample: 234 days | Lift: 1.43x
     overnight_touched_r1=True, ny_orb_first_break=high, orb_935_first_break=high
     → BULLISH NY: 183/234 days (78.2% accuracy)

#4 | Score: 73.4 | Conf: 77.8% | Edge: 27.8% |

In [4]:
## Top 50 Bearish Patterns - NY Session (9:30-4pm)

import pandas as pd
import numpy as np

input_dir = r"C:\Users\tammm\NQ and ES analysis"

# Load pattern files
print("Loading patterns for NY session...")
df_945 = pd.read_csv(f'{input_dir}\\smart_rules_945am.csv')
df_1030 = pd.read_csv(f'{input_dir}\\smart_rules_1030am.csv')

# Baseline from previous analysis
BASELINE_BULLISH = 0.546
BASELINE_BEARISH = 0.454

print("\n" + "="*130)
print("TRADEABLE BEARISH PATTERNS - NY SESSION (9:30-4PM)")
print("="*130)
print("Minimum sample size: 50 days")
print("Minimum confidence: 65%")
print("Outcome: NY session direction (9:30-4pm)")
print()

# Filter criteria
MIN_SAMPLE = 50
MIN_CONFIDENCE = 0.65

# Calculate actionability score
def calc_score(row):
    edge = abs(row['confidence'] - 0.5) * 100
    confidence_score = min(edge * 2, 100)
    sample = row['condition_count']
    if sample < 50:
        freq_score = (sample / 50) * 50
    else:
        freq_score = min(50 + (np.log2(sample / 50) * 25), 100)
    return round((confidence_score * 0.6) + (freq_score * 0.4), 1)

df_945['score'] = df_945.apply(calc_score, axis=1)
df_1030['score'] = df_1030.apply(calc_score, axis=1)

# Calculate true edge
df_945['edge_pct'] = abs(df_945['confidence'] - 0.5) * 100
df_1030['edge_pct'] = abs(df_1030['confidence'] - 0.5) * 100

# Filter for tradeable bearish patterns
bearish_945 = df_945[
    (df_945['outcome'] == 'bearish') & 
    (df_945['condition_count'] >= MIN_SAMPLE) &
    (df_945['confidence'] >= MIN_CONFIDENCE)
].sort_values('score', ascending=False)

bearish_1030 = df_1030[
    (df_1030['outcome'] == 'bearish') & 
    (df_1030['condition_count'] >= MIN_SAMPLE) &
    (df_1030['confidence'] >= MIN_CONFIDENCE)
].sort_values('score', ascending=False)

# Display function
def display_patterns(df, title, n=50):
    print("\n" + "="*130)
    print(f"{title}")
    print("="*130)
    
    if len(df) == 0:
        print("No tradeable bearish patterns found with these criteria")
        return
    
    print(f"Found {len(df)} tradeable bearish patterns\n")
    
    for i, (idx, row) in enumerate(df.head(n).iterrows()):
        if i >= n:
            break
            
        # Build condition string
        cond = f"{row['f1']}={row['v1']}"
        if pd.notna(row.get('f2')):
            cond += f", {row['f2']}={row['v2']}"
        if pd.notna(row.get('f3')):
            cond += f", {row['f3']}={row['v3']}"
        
        # Calculate vs baseline
        true_edge = (row['confidence'] - BASELINE_BEARISH) * 100
        
        print(f"\n#{i+1} | Score: {row['score']:.1f} | Conf: {row['confidence']*100:.1f}% | "
              f"Edge: {row['edge_pct']:.1f}% | True Edge: {true_edge:+.1f}%")
        print(f"     Sample: {row['condition_count']} days | Lift: {row['lift']:.2f}x")
        print(f"     {cond}")
        print(f"     → BEARISH NY: {row['count']}/{row['condition_count']} days "
              f"({row['confidence']*100:.1f}% accuracy)")

# Display results
display_patterns(bearish_945, "TRADEABLE BEARISH PATTERNS - 9:45AM ENTRY", 50)
display_patterns(bearish_1030, "TRADEABLE BEARISH PATTERNS - 10:30AM ENTRY", 50)

# Summary statistics
print("\n" + "="*130)
print("SUMMARY - TRADEABLE BEARISH PATTERNS")
print("="*130)

print("\n9:45AM ENTRY:")
print(f"  Total bearish patterns: {len(df_945[df_945['outcome'] == 'bearish']):,}")
print(f"  Tradeable (50+ days, 65%+ conf): {len(bearish_945):,}")
if len(bearish_945) > 0:
    print(f"  Best confidence: {bearish_945['confidence'].max()*100:.1f}%")
    print(f"  Best score: {bearish_945['score'].max():.1f}")
    print(f"  Average confidence: {bearish_945['confidence'].mean()*100:.1f}%")
    print(f"  Average sample size: {bearish_945['condition_count'].mean():.0f} days")

print("\n10:30AM ENTRY:")
print(f"  Total bearish patterns: {len(df_1030[df_1030['outcome'] == 'bearish']):,}")
print(f"  Tradeable (50+ days, 65%+ conf): {len(bearish_1030):,}")
if len(bearish_1030) > 0:
    print(f"  Best confidence: {bearish_1030['confidence'].max()*100:.1f}%")
    print(f"  Best score: {bearish_1030['score'].max():.1f}")
    print(f"  Average confidence: {bearish_1030['confidence'].mean()*100:.1f}%")
    print(f"  Average sample size: {bearish_1030['condition_count'].mean():.0f} days")

# Compare to bullish
bullish_945 = df_945[
    (df_945['outcome'] == 'bullish') & 
    (df_945['condition_count'] >= MIN_SAMPLE) &
    (df_945['confidence'] >= MIN_CONFIDENCE)
]

bullish_1030 = df_1030[
    (df_1030['outcome'] == 'bullish') & 
    (df_1030['condition_count'] >= MIN_SAMPLE) &
    (df_1030['confidence'] >= MIN_CONFIDENCE)
]

print("\n" + "="*130)
print("COMPARISON: BEARISH vs BULLISH (Tradeable Patterns Only)")
print("="*130)

print("\n9:45AM:")
print(f"  Tradeable bearish: {len(bearish_945):,} patterns | Best conf: {bearish_945['confidence'].max()*100:.1f}%" if len(bearish_945) > 0 else "  Tradeable bearish: 0 patterns")
print(f"  Tradeable bullish: {len(bullish_945):,} patterns | Best conf: {bullish_945['confidence'].max()*100:.1f}%" if len(bullish_945) > 0 else "  Tradeable bullish: 0 patterns")

print("\n10:30AM:")
print(f"  Tradeable bearish: {len(bearish_1030):,} patterns | Best conf: {bearish_1030['confidence'].max()*100:.1f}%" if len(bearish_1030) > 0 else "  Tradeable bearish: 0 patterns")
print(f"  Tradeable bullish: {len(bullish_1030):,} patterns | Best conf: {bullish_1030['confidence'].max()*100:.1f}%" if len(bullish_1030) > 0 else "  Tradeable bullish: 0 patterns")

# Export tradeable patterns
if len(bearish_945) > 0:
    bearish_945.head(50).to_csv(f'{input_dir}\\tradeable_bearish_945am.csv', index=False)
    print(f"\n✓ Exported: tradeable_bearish_945am.csv ({len(bearish_945.head(50))} patterns)")

if len(bearish_1030) > 0:
    bearish_1030.head(50).to_csv(f'{input_dir}\\tradeable_bearish_1030am.csv', index=False)
    print(f"✓ Exported: tradeable_bearish_1030am.csv ({len(bearish_1030.head(50))} patterns)")

print("\n" + "="*130)
print("KEY INSIGHTS")
print("="*130)
print("\n1. Bearish patterns are RARE because:")
print("   - Market baseline is 54.6% bullish (bull market period)")
print("   - Strong bearish setups don't occur frequently")
print("   - Most bearish patterns have small sample sizes (< 50 days)")

print("\n2. What makes a good bearish pattern:")
print("   - >70% bearish (30% bullish) = 15.4% edge over baseline")
print("   - 50+ occurrences = tradeable frequency")
print("   - High lift (>1.3x) = true predictive power")

print("\n3. Expected baseline for common bearish setups:")
print("   - IB bearish + ORB low: 69.2% bearish (30.8% bullish)")
print("   - Touched R1 + IB bearish: 71.9% bearish (28.1% bullish)")
print("   - ORB bearish + breaks low: 66.0% bearish (34.0% bullish)")

if len(bearish_945) == 0 and len(bearish_1030) == 0:
    print("\n⚠️  NO TRADEABLE BEARISH PATTERNS FOUND!")
    print("This suggests the pattern discovery needs adjustment.")
    print("Try lowering minimum confidence to 60% or minimum sample to 30 days.")

print("\n" + "="*130)

Loading patterns for NY session...

TRADEABLE BEARISH PATTERNS - NY SESSION (9:30-4PM)
Minimum sample size: 50 days
Minimum confidence: 65%
Outcome: NY session direction (9:30-4pm)


TRADEABLE BEARISH PATTERNS - 9:45AM ENTRY
Found 261 tradeable bearish patterns


#1 | Score: 65.4 | Conf: 71.2% | Edge: 21.2% | True Edge: +25.8%
     Sample: 215 days | Lift: 1.57x
     overnight_touched_r1=True, ny_orb_direction=bearish, ny_orb_first_break=low
     → BEARISH NY: 153/215 days (71.2% accuracy)

#2 | Score: 65.1 | Conf: 72.2% | Edge: 22.2% | True Edge: +26.8%
     Sample: 180 days | Lift: 1.56x
     ny_open_vs_prev_day=above_high, asia_direction=bullish, ny_orb_first_break=low
     → BEARISH NY: 130/180 days (72.2% accuracy)

#3 | Score: 64.5 | Conf: 71.1% | Edge: 21.1% | True Edge: +25.7%
     Sample: 190 days | Lift: 1.57x
     overnight_touched_r1=True, ny_orb_first_break=low, orb_935_direction=bearish
     → BEARISH NY: 135/190 days (71.1% accuracy)

#4 | Score: 64.1 | Conf: 70.1% | Edg

In [5]:
## Compare Bullish vs Bearish Patterns - NY Session (9:30-4pm)

import pandas as pd
import numpy as np

input_dir = r"C:\Users\tammm\NQ and ES analysis"

# Load pattern files
print("Loading patterns for NY session...")
df_945 = pd.read_csv(f'{input_dir}\\smart_rules_945am.csv')
df_1030 = pd.read_csv(f'{input_dir}\\smart_rules_1030am.csv')

# Baseline
BASELINE_BULLISH = 0.546
BASELINE_BEARISH = 0.454

print("\n" + "="*130)
print("BULLISH vs BEARISH PATTERN COMPARISON - NY SESSION (9:30-4PM)")
print("="*130)
print("Outcome: NY session direction (9:30-4pm)")
print()

# Calculate scores
def calc_score(row):
    edge = abs(row['confidence'] - 0.5) * 100
    confidence_score = min(edge * 2, 100)
    sample = row['condition_count']
    if sample < 50:
        freq_score = (sample / 50) * 50
    else:
        freq_score = min(50 + (np.log2(sample / 50) * 25), 100)
    return round((confidence_score * 0.6) + (freq_score * 0.4), 1)

df_945['score'] = df_945.apply(calc_score, axis=1)
df_1030['score'] = df_1030.apply(calc_score, axis=1)
df_945['edge_pct'] = abs(df_945['confidence'] - 0.5) * 100
df_1030['edge_pct'] = abs(df_1030['confidence'] - 0.5) * 100

# ============================================================
# 9:45AM ENTRY COMPARISON
# ============================================================
print("="*130)
print("9:45AM ENTRY - OVERALL STATISTICS")
print("="*130)

bullish_945 = df_945[df_945['outcome'] == 'bullish']
bearish_945 = df_945[df_945['outcome'] == 'bearish']

print(f"\nTotal patterns: {len(df_945):,}")
print(f"  Bullish patterns: {len(bullish_945):,} ({len(bullish_945)/len(df_945)*100:.1f}%)")
print(f"  Bearish patterns: {len(bearish_945):,} ({len(bearish_945)/len(df_945)*100:.1f}%)")

# By feature count
print("\n" + "="*130)
print("9:45AM - PATTERNS BY COMPLEXITY")
print("="*130)

for feat_count in [1, 2, 3]:
    bull_feat = bullish_945[bullish_945['feature_count'] == feat_count]
    bear_feat = bearish_945[bearish_945['feature_count'] == feat_count]
    
    print(f"\n{feat_count}-Feature Patterns:")
    print(f"  Bullish: {len(bull_feat):,} patterns")
    if len(bull_feat) > 0:
        print(f"    Best conf: {bull_feat['confidence'].max()*100:.1f}%")
        print(f"    Avg conf: {bull_feat['confidence'].mean()*100:.1f}%")
        print(f"    Avg sample: {bull_feat['condition_count'].mean():.0f} days")
    
    print(f"  Bearish: {len(bear_feat):,} patterns")
    if len(bear_feat) > 0:
        print(f"    Best conf: {bear_feat['confidence'].max()*100:.1f}%")
        print(f"    Avg conf: {bear_feat['confidence'].mean()*100:.1f}%")
        print(f"    Avg sample: {bear_feat['condition_count'].mean():.0f} days")

# Tradeable patterns
print("\n" + "="*130)
print("9:45AM - TRADEABLE PATTERNS (50+ days, 65%+ confidence)")
print("="*130)

MIN_SAMPLE = 50
MIN_CONFIDENCE = 0.65

tradeable_bull_945 = bullish_945[(bullish_945['condition_count'] >= MIN_SAMPLE) & (bullish_945['confidence'] >= MIN_CONFIDENCE)]
tradeable_bear_945 = bearish_945[(bearish_945['condition_count'] >= MIN_SAMPLE) & (bearish_945['confidence'] >= MIN_CONFIDENCE)]

print(f"\nBullish: {len(tradeable_bull_945):,} tradeable patterns")
if len(tradeable_bull_945) > 0:
    print(f"  Best confidence: {tradeable_bull_945['confidence'].max()*100:.1f}%")
    print(f"  Best score: {tradeable_bull_945['score'].max():.1f}")
    print(f"  Avg sample: {tradeable_bull_945['condition_count'].mean():.0f} days")

print(f"\nBearish: {len(tradeable_bear_945):,} tradeable patterns")
if len(tradeable_bear_945) > 0:
    print(f"  Best confidence: {tradeable_bear_945['confidence'].max()*100:.1f}%")
    print(f"  Best score: {tradeable_bear_945['score'].max():.1f}")
    print(f"  Avg sample: {tradeable_bear_945['condition_count'].mean():.0f} days")

# ============================================================
# 10:30AM ENTRY COMPARISON
# ============================================================
print("\n\n" + "="*130)
print("10:30AM ENTRY - OVERALL STATISTICS")
print("="*130)

bullish_1030 = df_1030[df_1030['outcome'] == 'bullish']
bearish_1030 = df_1030[df_1030['outcome'] == 'bearish']

print(f"\nTotal patterns: {len(df_1030):,}")
print(f"  Bullish patterns: {len(bullish_1030):,} ({len(bullish_1030)/len(df_1030)*100:.1f}%)")
print(f"  Bearish patterns: {len(bearish_1030):,} ({len(bearish_1030)/len(df_1030)*100:.1f}%)")

# By feature count
print("\n" + "="*130)
print("10:30AM - PATTERNS BY COMPLEXITY")
print("="*130)

for feat_count in [1, 2, 3]:
    bull_feat = bullish_1030[bullish_1030['feature_count'] == feat_count]
    bear_feat = bearish_1030[bearish_1030['feature_count'] == feat_count]
    
    print(f"\n{feat_count}-Feature Patterns:")
    print(f"  Bullish: {len(bull_feat):,} patterns")
    if len(bull_feat) > 0:
        print(f"    Best conf: {bull_feat['confidence'].max()*100:.1f}%")
        print(f"    Avg conf: {bull_feat['confidence'].mean()*100:.1f}%")
        print(f"    Avg sample: {bull_feat['condition_count'].mean():.0f} days")
    
    print(f"  Bearish: {len(bear_feat):,} patterns")
    if len(bear_feat) > 0:
        print(f"    Best conf: {bear_feat['confidence'].max()*100:.1f}%")
        print(f"    Avg conf: {bear_feat['confidence'].mean()*100:.1f}%")
        print(f"    Avg sample: {bear_feat['condition_count'].mean():.0f} days")

# Tradeable patterns
print("\n" + "="*130)
print("10:30AM - TRADEABLE PATTERNS (50+ days, 65%+ confidence)")
print("="*130)

tradeable_bull_1030 = bullish_1030[(bullish_1030['condition_count'] >= MIN_SAMPLE) & (bullish_1030['confidence'] >= MIN_CONFIDENCE)]
tradeable_bear_1030 = bearish_1030[(bearish_1030['condition_count'] >= MIN_SAMPLE) & (bearish_1030['confidence'] >= MIN_CONFIDENCE)]

print(f"\nBullish: {len(tradeable_bull_1030):,} tradeable patterns")
if len(tradeable_bull_1030) > 0:
    print(f"  Best confidence: {tradeable_bull_1030['confidence'].max()*100:.1f}%")
    print(f"  Best score: {tradeable_bull_1030['score'].max():.1f}")
    print(f"  Avg sample: {tradeable_bull_1030['condition_count'].mean():.0f} days")

print(f"\nBearish: {len(tradeable_bear_1030):,} tradeable patterns")
if len(tradeable_bear_1030) > 0:
    print(f"  Best confidence: {tradeable_bear_1030['confidence'].max()*100:.1f}%")
    print(f"  Best score: {tradeable_bear_1030['score'].max():.1f}")
    print(f"  Avg sample: {tradeable_bear_1030['condition_count'].mean():.0f} days")

# ============================================================
# SIDE-BY-SIDE TOP 10
# ============================================================
print("\n\n" + "="*130)
print("TOP 10 PATTERNS - SIDE BY SIDE (9:45AM ENTRY)")
print("="*130)

print("\n" + "-"*130)
print(f"{'BULLISH':<65} | {'BEARISH':<64}")
print("-"*130)

# Sort by score
bull_sorted = bullish_945.sort_values('score', ascending=False).head(10)
bear_sorted = bearish_945.sort_values('score', ascending=False).head(10)

max_rows = max(len(bull_sorted), len(bear_sorted))

for i in range(max_rows):
    # Bullish side
    if i < len(bull_sorted):
        bull_row = bull_sorted.iloc[i]
        bull_cond = f"{bull_row['f1']}={bull_row['v1']}"
        if pd.notna(bull_row.get('f2')):
            bull_cond += f",{bull_row['f2']}={bull_row['v2']}"
        bull_text = f"{bull_row['confidence']*100:.1f}% ({bull_row['condition_count']}d) {bull_cond[:30]}"
    else:
        bull_text = ""
    
    # Bearish side
    if i < len(bear_sorted):
        bear_row = bear_sorted.iloc[i]
        bear_cond = f"{bear_row['f1']}={bear_row['v1']}"
        if pd.notna(bear_row.get('f2')):
            bear_cond += f",{bear_row['f2']}={bear_row['v2']}"
        bear_text = f"{bear_row['confidence']*100:.1f}% ({bear_row['condition_count']}d) {bear_cond[:30]}"
    else:
        bear_text = ""
    
    print(f"{bull_text:<65} | {bear_text:<64}")

# ============================================================
# KEY INSIGHTS
# ============================================================
print("\n\n" + "="*130)
print("KEY INSIGHTS")
print("="*130)

print("\n1. PATTERN BALANCE:")
if len(bullish_945) > len(bearish_945) * 1.5:
    print("   → Significantly more bullish patterns found")
    print("   → NY session has bullish bias (54.6% baseline)")
elif len(bearish_945) > len(bullish_945) * 1.5:
    print("   → Significantly more bearish patterns found")
else:
    print("   → Relatively balanced bullish/bearish patterns")

print("\n2. QUALITY COMPARISON (9:45am):")
if len(tradeable_bull_945) > 0 and len(tradeable_bear_945) > 0:
    bull_best = tradeable_bull_945['confidence'].max() * 100
    bear_best = tradeable_bear_945['confidence'].max() * 100
    print(f"   → Best bullish: {bull_best:.1f}%")
    print(f"   → Best bearish: {bear_best:.1f}%")
    if bull_best > bear_best + 5:
        print("   → Bullish patterns have higher confidence")
    elif bear_best > bull_best + 5:
        print("   → Bearish patterns have higher confidence")
    else:
        print("   → Similar quality for both directions")
elif len(tradeable_bull_945) > 0:
    print("   → Only bullish patterns meet tradeable criteria")
elif len(tradeable_bear_945) > 0:
    print("   → Only bearish patterns meet tradeable criteria")

print("\n3. COMPLEXITY MATTERS:")
print("   → 3-feature patterns add 12-14% incremental edge on average")
print("   → But they reduce sample size by 40-60%")
print("   → Best for high-confidence setups, not daily trading")

print("\n4. USAGE:")
print("   → Trade 1-2 feature patterns daily (high frequency)")
print("   → Save 3-feature patterns for high-conviction trades")
print("   → Bullish patterns more reliable in current market")

print("\n" + "="*130)

Loading patterns for NY session...

BULLISH vs BEARISH PATTERN COMPARISON - NY SESSION (9:30-4PM)
Outcome: NY session direction (9:30-4pm)

9:45AM ENTRY - OVERALL STATISTICS

Total patterns: 15,594
  Bullish patterns: 11,162 (71.6%)
  Bearish patterns: 4,432 (28.4%)

9:45AM - PATTERNS BY COMPLEXITY

1-Feature Patterns:
  Bullish: 11 patterns
    Best conf: 70.1%
    Avg conf: 61.7%
    Avg sample: 703 days
  Bearish: 3 patterns
    Best conf: 61.9%
    Avg conf: 61.1%
    Avg sample: 1004 days

2-Feature Patterns:
  Bullish: 847 patterns
    Best conf: 100.0%
    Avg conf: 64.0%
    Avg sample: 363 days
  Bearish: 286 patterns
    Best conf: 85.7%
    Avg conf: 62.3%
    Avg sample: 474 days

3-Feature Patterns:
  Bullish: 10,304 patterns
    Best conf: 80.6%
    Avg conf: 66.4%
    Avg sample: 395 days
  Bearish: 4,143 patterns
    Best conf: 72.2%
    Avg conf: 61.6%
    Avg sample: 365 days

9:45AM - TRADEABLE PATTERNS (50+ days, 65%+ confidence)

Bullish: 6,817 tradeable patterns
 